# PRC Data Challenge 2026 — V17.2 FINAL ONE-RUN
## Operational Features + Single LightGBM (max 1200 trees)

This is the fast production version of V17.

The expensive multi-model ensemble has been removed.

### Known-anchor model

```text
V17 operational / queue / recurrence features
                |
        ONE LightGBM
        max 1200 trees
                |
       NM anchor + residual
```

### Kept from V17

- airport × stand × runway features
- operator / recurring-flight features
- NM match-quality features
- runway arrival/departure sequence features
- runway gap / queue features
- off-block → takeoff movement counts
- peer-imputed missing-anchor model
- synthetic missing-anchor augmentation
- full-2025 production training
- official submission generation

No CatBoost. No DNN. No FT-Transformer. No multi-LightGBM ensemble.

## FINAL RUN MODE

This notebook is configured with `RUN_FINAL=True`.

A single top-to-bottom execution will:

1. run the October development validation,
2. freeze the selected LightGBM iteration count (never above 1200),
3. freeze the missing-anchor configuration,
4. retrain on all 2025 training departures,
5. predict every ranking departure,
6. build the official submission parquet,
7. validate the saved parquet by reading it back,
8. print the exact filename, local path, and OpenSky bucket.

Do not change the modeling configuration during this run.


In [ ]:
#@title 1. V17 setup
# In Colab, install if needed:
# !pip install -q lightgbm scipy pyarrow pandas numpy scikit-learn matplotlib

import os
import gc
import math
import shutil
import zipfile
import warnings
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lightgbm import LGBMRegressor, early_stopping as lgb_early_stopping, log_evaluation as lgb_log_evaluation
import lightgbm as lgb
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

if os.environ.get("PRC_ROOT"):
    # Local runs: PRC_ROOT=/path/to/dir with data/raw/*.parquet inside.
    ROOT = Path(os.environ["PRC_ROOT"])
elif Path("/workspace/data/raw").exists():
    ROOT = Path("/workspace")
else:
    ROOT = Path("/content")

RAW_DIR = ROOT / "data" / "raw"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
RUN_DECEMBER = False
RUN_FINAL = True

# V14 missing branch switches.
RUN_PEER_MISSING_BRANCH = True
RUN_PEER_MISSING_AUDIT = False

# V17 synthetic missing-anchor augmentation.
PSEUDO_MISSING_ROWS = 120_000
PSEUDO_MISSING_REAL_WEIGHT = 4.0
PSEUDO_MISSING_SYNTH_WEIGHT = 1.0

# Old V9 direct missing branch is no longer retrained in V14.
RUN_MISSING_BRANCH = False
RUN_MISSING_AUDIT = False

GRAPH_EMBED_DIM = 8
GRAPH_MIN_EDGE_COUNT = 3

print("ROOT:", ROOT)
print("Data directory:", RAW_DIR)
print("Output directory:", OUTPUT_DIR)
print("RUN_DECEMBER:", RUN_DECEMBER)
print("RUN_FINAL:", RUN_FINAL)
print("RUN_PEER_MISSING_BRANCH:", RUN_PEER_MISSING_BRANCH)
print("RUN_PEER_MISSING_AUDIT:", RUN_PEER_MISSING_AUDIT)
print("RUN_MISSING_BRANCH:", RUN_MISSING_BRANCH)

print("LightGBM version:", lgb.__version__)



# Lightweight system-RAM diagnostics.
def report_ram(label=""):
    try:
        import psutil
    except ImportError:
        import subprocess, sys
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "psutil"],
            check=True
        )
        import psutil

    vm = psutil.virtual_memory()
    used_gb = (vm.total - vm.available) / (1024**3)
    total_gb = vm.total / (1024**3)
    avail_gb = vm.available / (1024**3)

    print(
        f"[RAM] {label}: "
        f"used={used_gb:.2f} GB / {total_gb:.2f} GB, "
        f"available={avail_gb:.2f} GB"
    )

report_ram("after setup")




In [ ]:
#@title 2. Locate raw files (local) or upload in Colab
if RAW_DIR.exists() and len(list(RAW_DIR.glob("*.parquet"))) >= 14:
    print("Using existing raw files:", RAW_DIR)
else:
    RAW_DIR.mkdir(parents=True, exist_ok=True)

    # Colab-only upload path.
    try:
        from google.colab import files
        uploaded = files.upload()

        upload_dir = ROOT / "upload"
        extract_dir = ROOT / "extracted"

        for d in [upload_dir, extract_dir]:
            if d.exists():
                shutil.rmtree(d)
            d.mkdir(parents=True, exist_ok=True)

        for name, data in uploaded.items():
            p = upload_dir / Path(name).name
            p.write_bytes(data)

        for p in upload_dir.iterdir():
            if p.suffix.lower() == ".zip":
                with zipfile.ZipFile(p, "r") as zf:
                    zf.extractall(extract_dir / p.stem)

        for base in [upload_dir, extract_dir]:
            for p in base.rglob("*.parquet"):
                dest = RAW_DIR / p.name
                if not dest.exists():
                    shutil.copy2(p, dest)

    except ImportError:
        raise FileNotFoundError(
            f"Put the challenge parquet files in {RAW_DIR}"
        )

print("\nAvailable parquet files:")
for p in sorted(RAW_DIR.glob("*.parquet")):
    print(f"  {p.name:52s} {p.stat().st_size/1024**2:8.2f} MiB")

In [ ]:
#@title 3. Verify official files and select ranking stage
TRAIN_FILES = [
    "training_2025-01-01_2025-02-01.parquet",
    "training_2025-02-01_2025-03-01.parquet",
    "training_2025-03-01_2025-04-01.parquet",
    "training_2025-04-01_2025-05-01.parquet",
    "training_2025-05-01_2025-06-01.parquet",
    "training_2025-06-01_2025-07-01.parquet",
    "training_2025-07-01_2025-08-01.parquet",
    "training_2025-08-01_2025-09-01.parquet",
    "training_2025-09-01_2025-10-01.parquet",
    "training_2025-10-01_2025-11-01.parquet",
    "training_2025-11-01_2025-12-01.parquet",
    "training_2025-12-01_2026-01-01.parquet",
]

missing_train = [f for f in TRAIN_FILES if not (RAW_DIR / f).exists()]
assert not missing_train, f"Missing training files: {missing_train}"

has_final = (
    (RAW_DIR / "final_ranking.parquet").exists()
    and (RAW_DIR / "final_submitting.parquet").exists()
)

if has_final:
    RANKING_FILE = "final_ranking.parquet"
    TEMPLATE_FILE = "final_submitting.parquet"
    SUBMISSION_SUFFIX = "final"
    print("✓ FINAL stage files detected.")
else:
    RANKING_FILE = "ranking.parquet"
    TEMPLATE_FILE = "submitting.parquet"
    SUBMISSION_SUFFIX = "v3"
    assert (RAW_DIR / RANKING_FILE).exists()
    assert (RAW_DIR / TEMPLATE_FILE).exists()
    print("✓ Standard ranking stage files detected.")

print("Ranking file :", RANKING_FILE)
print("Template file:", TEMPLATE_FILE)

# ============================================================
# OFFICIAL SUBMISSION NAMING
# ============================================================
TEAM_NAME = "enthusiastic-yawn"

# Increment this for each regular leaderboard upload.
SUBMISSION_VERSION = 1

if SUBMISSION_SUFFIX == "final":
    OFFICIAL_SUBMISSION_NAME = f"{TEAM_NAME}_final.parquet"
else:
    OFFICIAL_SUBMISSION_NAME = f"{TEAM_NAME}_v{SUBMISSION_VERSION}.parquet"

print("Official output filename:", OFFICIAL_SUBMISSION_NAME)


# Team submission bucket from OpenSky acceptance email.
SUBMISSION_BUCKET = "prc-2026-enthusiastic-yawn"


In [ ]:
#@title 4. Load all 2025 movements
parts = []

for i, fname in enumerate(TRAIN_FILES, 1):
    df = pd.read_parquet(RAW_DIR / fname)
    parts.append(df)
    print(f"{i:02d}/12 {fname}: {len(df):,} movements")

train_all = pd.concat(parts, ignore_index=True)
del parts
gc.collect()

ranking_all = pd.read_parquet(RAW_DIR / RANKING_FILE)
template = pd.read_parquet(RAW_DIR / TEMPLATE_FILE)

print("\nTraining movements:", f"{len(train_all):,}")
print("Ranking movements :", f"{len(ranking_all):,}")
print("Template rows     :", f"{len(template):,}")

In [ ]:
#@title 5. V17 traffic / congestion context builder
def to_utc(s):
    return pd.to_datetime(
        s,
        utc=True,
        errors="coerce",
    )


def safe_divide(a, b):
    a = np.asarray(
        a,
        dtype=np.float32,
    )
    b = np.asarray(
        b,
        dtype=np.float32,
    )

    return np.divide(
        a,
        b,
        out=np.zeros_like(a),
        where=b != 0,
    )


def build_traffic_context(df):
    """
    Batch-level traffic context built only from supplied movement rows.

    These are label-free features. They use MVT_TIME/PHASE/RUNWAY from the
    same dataset split, which are available in ranking data.
    """
    out = df.copy()

    t = to_utc(
        out["MVT_TIME_UTC_mvt"]
    )

    phase = (
        out["PHASE_mvt"]
        .astype(str)
    )

    airport = np.where(
        phase.eq("DEP"),
        out["ADEP_mvt"]
        .fillna("__MISSING__")
        .astype(str),
        out["ADES_mvt"]
        .fillna("__MISSING__")
        .astype(str),
    )

    tmp = pd.DataFrame({
        "_airport": airport,
        "_phase": phase,
        "_time": t,
        "_runway": (
            out["RUNWAY_mvt"]
            .fillna("__MISSING__")
            .astype(str)
        ),
    }, index=out.index)

    tmp["_is_dep"] = (
        tmp["_phase"]
        .eq("DEP")
        .astype("int8")
    )

    tmp["_is_arr"] = (
        tmp["_phase"]
        .eq("ARR")
        .astype("int8")
    )

    # Airport load at three resolutions.
    for mins in [
        5,
        15,
        30,
    ]:
        b = f"_bin{mins}"

        tmp[b] = (
            tmp["_time"]
            .dt.floor(
                f"{mins}min"
            )
        )

        grp = tmp.groupby(
            [
                "_airport",
                b,
            ],
            sort=False,
        )

        dep = (
            grp["_is_dep"]
            .transform("sum")
            .astype("float32")
        )

        arr = (
            grp["_is_arr"]
            .transform("sum")
            .astype("float32")
        )

        total = (
            dep
            + arr
        ).astype(
            "float32"
        )

        out[
            f"airport_dep_{mins}m"
        ] = dep

        out[
            f"airport_arr_{mins}m"
        ] = arr

        out[
            f"airport_total_{mins}m"
        ] = total

        out[
            f"airport_dep_minus_arr_{mins}m"
        ] = (
            dep
            - arr
        ).astype(
            "float32"
        )

        out[
            f"airport_dep_fraction_{mins}m"
        ] = safe_divide(
            dep,
            total,
        )

    # Runway-level 15-minute pressure.
    tmp["_bin15_runway"] = (
        tmp["_time"]
        .dt.floor(
            "15min"
        )
    )

    rgrp = tmp.groupby(
        [
            "_airport",
            "_runway",
            "_bin15_runway",
        ],
        sort=False,
    )

    runway_total = (
        rgrp["_is_dep"]
        .transform("size")
        .astype("float32")
    )

    runway_dep = (
        rgrp["_is_dep"]
        .transform("sum")
        .astype("float32")
    )

    runway_arr = (
        rgrp["_is_arr"]
        .transform("sum")
        .astype("float32")
    )

    out[
        "runway_total_15m"
    ] = runway_total

    out[
        "runway_dep_15m"
    ] = runway_dep

    out[
        "runway_arr_15m"
    ] = runway_arr

    out[
        "runway_dep_minus_arr_15m"
    ] = (
        runway_dep
        - runway_arr
    ).astype(
        "float32"
    )

    out[
        "runway_dep_fraction_15m"
    ] = safe_divide(
        runway_dep,
        runway_total,
    )

    out[
        "runway_dep_share_15m"
    ] = safe_divide(
        runway_dep,
        out[
            "airport_dep_15m"
        ],
    )

    out[
        "runway_total_share_15m"
    ] = safe_divide(
        runway_total,
        out[
            "airport_total_15m"
        ],
    )

    # Number of distinct departure runways active in the airport/time bin.
    dep_tmp = tmp.loc[
        tmp["_is_dep"].eq(1),
        [
            "_airport",
            "_bin15_runway",
            "_runway",
        ],
    ]

    active_map = (
        dep_tmp.groupby(
            [
                "_airport",
                "_bin15_runway",
            ],
            sort=False,
        )["_runway"]
        .nunique()
    )

    active_key = pd.MultiIndex.from_arrays(
        [
            tmp["_airport"],
            tmp["_bin15_runway"],
        ]
    )

    out[
        "airport_active_dep_runways_15m"
    ] = (
        active_map.reindex(
            active_key
        )
        .fillna(0)
        .to_numpy(
            dtype=np.float32
        )
    )

    return out


print(
    "Building 2025 traffic context..."
)

train_all = build_traffic_context(
    train_all
)

gc.collect()

print(
    "Building ranking traffic context..."
)

ranking_all = build_traffic_context(
    ranking_all
)

gc.collect()

print(
    "✓ V17 traffic context built."
)

## Dynamic temporal graph features

These features create a local flight-interaction graph directly from movement times.

A movement is connected to nearby movements when they share:

- the same airport, or
- the same airport + runway

The features use no target labels.

**Important:** centered and future-neighbor features are retrospective. They are appropriate for this competition-style reconstruction pipeline, but they should not be interpreted as pre-pushback operational forecasting features.

In [ ]:
#@title 5b. V17 dynamic runway-sequence / temporal graph helpers

def _rolling_previous_gap_stats(
    t,
):
    """
    For each event, summarize only gaps strictly BEFORE that event.
    No target labels are used.
    """
    n = len(
        t
    )

    mean3 = np.full(
        n,
        np.nan,
        dtype=np.float32,
    )

    mean5 = np.full(
        n,
        np.nan,
        dtype=np.float32,
    )

    std5 = np.full(
        n,
        np.nan,
        dtype=np.float32,
    )

    if n <= 1:
        return (
            mean3,
            mean5,
            std5,
        )

    gaps = np.diff(
        t
    ).astype(
        np.float64
    )

    for i in range(
        1,
        n,
    ):
        g3 = gaps[
            max(
                0,
                i - 3,
            ):i
        ]

        g5 = gaps[
            max(
                0,
                i - 5,
            ):i
        ]

        if len(
            g3
        ):
            mean3[
                i
            ] = np.float32(
                np.mean(
                    g3
                )
            )

        if len(
            g5
        ):
            mean5[
                i
            ] = np.float32(
                np.mean(
                    g5
                )
            )

            std5[
                i
            ] = np.float32(
                np.std(
                    g5
                )
            )

    return (
        mean3,
        mean5,
        std5,
    )


def _local_temporal_arrays(
    times_sec,
    phase_dep=None,
):
    """
    Compute temporal/sequence arrays for ONE sorted airport or runway group.
    """
    n = len(
        times_sec
    )

    t = times_sec.astype(
        np.float64
    )

    result = {}

    dep = None
    arr = None
    dep_cum = None
    arr_cum = None

    if phase_dep is not None:
        dep = phase_dep.astype(
            np.int32
        )

        arr = (
            1
            - dep
        )

        dep_cum = np.empty(
            n + 1,
            dtype=np.int64,
        )

        arr_cum = np.empty(
            n + 1,
            dtype=np.int64,
        )

        dep_cum[
            0
        ] = 0

        arr_cum[
            0
        ] = 0

        np.cumsum(
            dep,
            out=dep_cum[
                1:
            ],
        )

        np.cumsum(
            arr,
            out=arr_cum[
                1:
            ],
        )

    for mins in [
        5,
        10,
        15,
    ]:
        w = (
            mins
            * 60.0
        )

        left = np.searchsorted(
            t,
            t - w,
            side="left",
        )

        right = np.searchsorted(
            t,
            t + w,
            side="right",
        )

        idx = np.arange(
            n,
            dtype=np.int64,
        )

        past = (
            idx
            - left
        )

        future = (
            right
            - idx
            - 1
        )

        result[
            f"deg_{mins}m"
        ] = (
            past
            + future
        ).astype(
            np.float32
        )

        result[
            f"past_{mins}m"
        ] = past.astype(
            np.float32
        )

        result[
            f"future_{mins}m"
        ] = future.astype(
            np.float32
        )

        if dep is not None:
            dep_all = (
                dep_cum[
                    right
                ]
                - dep_cum[
                    left
                ]
                - dep
            )

            arr_all = (
                arr_cum[
                    right
                ]
                - arr_cum[
                    left
                ]
                - arr
            )

            result[
                f"dep_{mins}m"
            ] = dep_all.astype(
                np.float32
            )

            result[
                f"arr_{mins}m"
            ] = arr_all.astype(
                np.float32
            )

            result[
                f"past_dep_{mins}m"
            ] = (
                dep_cum[
                    idx
                ]
                - dep_cum[
                    left
                ]
            ).astype(
                np.float32
            )

            result[
                f"past_arr_{mins}m"
            ] = (
                arr_cum[
                    idx
                ]
                - arr_cum[
                    left
                ]
            ).astype(
                np.float32
            )

            result[
                f"future_dep_{mins}m"
            ] = (
                dep_cum[
                    right
                ]
                - dep_cum[
                    idx + 1
                ]
            ).astype(
                np.float32
            )

            result[
                f"future_arr_{mins}m"
            ] = (
                arr_cum[
                    right
                ]
                - arr_cum[
                    idx + 1
                ]
            ).astype(
                np.float32
            )

    prev_gap = np.full(
        n,
        np.nan,
        dtype=np.float32,
    )

    next_gap = np.full(
        n,
        np.nan,
        dtype=np.float32,
    )

    if n > 1:
        d = np.diff(
            t
        ).astype(
            np.float32
        )

        prev_gap[
            1:
        ] = d

        next_gap[
            :-1
        ] = d

    result[
        "prev_gap_sec"
    ] = prev_gap

    result[
        "next_gap_sec"
    ] = next_gap

    (
        mean3,
        mean5,
        std5,
    ) = _rolling_previous_gap_stats(
        t
    )

    result[
        "prev_gap_mean3_sec"
    ] = mean3

    result[
        "prev_gap_mean5_sec"
    ] = mean5

    result[
        "prev_gap_std5_sec"
    ] = std5

    if dep is not None:
        prev_dep_gap = np.full(
            n,
            np.nan,
            dtype=np.float32,
        )

        prev_arr_gap = np.full(
            n,
            np.nan,
            dtype=np.float32,
        )

        consecutive_prev_dep = np.zeros(
            n,
            dtype=np.float32,
        )

        last_dep_time = np.nan
        last_arr_time = np.nan
        dep_streak = 0

        for i in range(
            n
        ):
            if np.isfinite(
                last_dep_time
            ):
                prev_dep_gap[
                    i
                ] = np.float32(
                    t[
                        i
                    ]
                    - last_dep_time
                )

            if np.isfinite(
                last_arr_time
            ):
                prev_arr_gap[
                    i
                ] = np.float32(
                    t[
                        i
                    ]
                    - last_arr_time
                )

            consecutive_prev_dep[
                i
            ] = np.float32(
                dep_streak
            )

            if dep[
                i
            ] == 1:
                last_dep_time = t[
                    i
                ]

                dep_streak += 1

            else:
                last_arr_time = t[
                    i
                ]

                dep_streak = 0

        result[
            "prev_dep_gap_sec"
        ] = prev_dep_gap

        result[
            "prev_arr_gap_sec"
        ] = prev_arr_gap

        result[
            "consecutive_prev_dep"
        ] = consecutive_prev_dep

    for tau in [
        300.0,
        900.0,
    ]:
        past_w = np.zeros(
            n,
            dtype=np.float32,
        )

        future_w = np.zeros(
            n,
            dtype=np.float32,
        )

        for i in range(
            1,
            n,
        ):
            decay = np.exp(
                -(
                    t[
                        i
                    ]
                    - t[
                        i - 1
                    ]
                )
                / tau
            )

            past_w[
                i
            ] = (
                decay
                * (
                    1.0
                    + past_w[
                        i - 1
                    ]
                )
            )

        for i in range(
            n - 2,
            -1,
            -1,
        ):
            decay = np.exp(
                -(
                    t[
                        i + 1
                    ]
                    - t[
                        i
                    ]
                )
                / tau
            )

            future_w[
                i
            ] = (
                decay
                * (
                    1.0
                    + future_w[
                        i + 1
                    ]
                )
            )

        result[
            f"exp_degree_tau{int(tau)}"
        ] = (
            past_w
            + future_w
        )

        result[
            f"exp_past_tau{int(tau)}"
        ] = past_w

    return result


def _selected_nm_anchor_seconds(
    departures,
):
    """
    Selected NM off-block timestamp in Unix seconds using the exact
    AOBT -> EOBT -> IOBT -> LOBT priority.
    """
    selected = pd.Series(
        pd.NaT,
        index=departures.index,
        dtype="datetime64[ns, UTC]",
    )

    for col in [
        "AOBT_3_flt",
        "EOBT_1_flt",
        "IOBT_flt",
        "LOBT_flt",
    ]:
        if col not in departures.columns:
            continue

        candidate = to_utc(
            departures[
                col
            ]
        )

        use = (
            selected.isna()
            & candidate.notna()
        )

        selected.loc[
            use
        ] = candidate.loc[
            use
        ]

    out = np.full(
        len(
            departures
        ),
        np.nan,
        dtype=np.float64,
    )

    good = selected.notna()

    if good.any():
        out[
            good.to_numpy()
        ] = (
            selected.loc[
                good
            ]
            .astype(
                "int64"
            )
            .to_numpy(
                dtype=np.float64
            )
            / 1e9
        )

    return out


def build_dynamic_features_for_departures(
    all_movements,
    departures,
):
    """
    V17 context:
      * airport temporal graph
      * runway temporal graph WITH phase-specific dep/arr features
      * local sequence/gap features
      * movement counts between NM off-block proxy and takeoff

    Uses supplied movement timestamps only; target taxi time is never used.
    """
    dep_index = departures.index
    n_target = len(
        departures
    )

    phase = (
        all_movements[
            "PHASE_mvt"
        ]
        .astype(str)
    )

    times = to_utc(
        all_movements[
            "MVT_TIME_UTC_mvt"
        ]
    )

    ctx = pd.DataFrame({
        "_orig_index": (
            all_movements.index
            .to_numpy()
        ),
        "_time_sec": (
            times.astype(
                "int64"
            )
            / 1e9
        ).to_numpy(),
        "_valid_time": (
            times.notna()
            .to_numpy()
        ),
        "_is_dep": (
            phase.eq(
                "DEP"
            )
            .astype(
                np.int8
            )
            .to_numpy()
        ),
        "_airport": np.where(
            phase.eq(
                "DEP"
            ),
            all_movements[
                "ADEP_mvt"
            ]
            .fillna(
                "__MISSING__"
            )
            .astype(str),
            all_movements[
                "ADES_mvt"
            ]
            .fillna(
                "__MISSING__"
            )
            .astype(str),
        ),
        "_runway_raw": (
            all_movements[
                "RUNWAY_mvt"
            ]
            .fillna(
                "__MISSING__"
            )
            .astype(str)
            .to_numpy()
        ),
    })

    ctx[
        "_runway_group"
    ] = (
        ctx[
            "_airport"
        ].astype(str)
        + "|"
        + ctx[
            "_runway_raw"
        ].astype(str)
    )

    target_pos = pd.Series(
        np.arange(
            n_target,
            dtype=np.int64,
        ),
        index=dep_index,
    )

    target_anchor_sec = pd.Series(
        _selected_nm_anchor_seconds(
            departures
        ),
        index=dep_index,
        dtype="float64",
    )

    airport_feature_keys = []

    runway_feature_keys = []

    for mins in [
        5,
        10,
        15,
    ]:
        common = [
            f"deg_{mins}m",
            f"past_{mins}m",
            f"future_{mins}m",
            f"dep_{mins}m",
            f"arr_{mins}m",
            f"past_dep_{mins}m",
            f"past_arr_{mins}m",
            f"future_dep_{mins}m",
            f"future_arr_{mins}m",
        ]

        airport_feature_keys += common
        runway_feature_keys += common

    sequence_keys = [
        "prev_gap_sec",
        "next_gap_sec",
        "prev_gap_mean3_sec",
        "prev_gap_mean5_sec",
        "prev_gap_std5_sec",
        "prev_dep_gap_sec",
        "prev_arr_gap_sec",
        "consecutive_prev_dep",
        "exp_degree_tau300",
        "exp_past_tau300",
        "exp_degree_tau900",
        "exp_past_tau900",
    ]

    airport_feature_keys += sequence_keys
    runway_feature_keys += sequence_keys

    buffers = {}

    for k in airport_feature_keys:
        buffers[
            f"tg_airport_{k}"
        ] = np.full(
            n_target,
            np.nan,
            dtype=np.float32,
        )

    for k in runway_feature_keys:
        buffers[
            f"tg_runway_{k}"
        ] = np.full(
            n_target,
            np.nan,
            dtype=np.float32,
        )

    interval_names = [
        "q_airport_movements_between",
        "q_airport_departures_between",
        "q_airport_arrivals_between",
        "q_runway_movements_between",
        "q_runway_departures_between",
        "q_runway_arrivals_between",
    ]

    for name in interval_names:
        buffers[
            name
        ] = np.full(
            n_target,
            np.nan,
            dtype=np.float32,
        )

    valid_ctx = ctx.loc[
        ctx[
            "_valid_time"
        ]
    ].copy()

    print(
        "  airport temporal/queue graph..."
    )

    for _, g in valid_ctx.groupby(
        "_airport",
        sort=False,
    ):
        g = g.sort_values(
            "_time_sec",
            kind="mergesort",
        )

        orig = g[
            "_orig_index"
        ].to_numpy()

        keep_mask = (
            pd.Index(
                orig
            )
            .isin(
                dep_index
            )
        )

        if not keep_mask.any():
            continue

        local_target = np.flatnonzero(
            keep_mask
        )

        target_orig = orig[
            keep_mask
        ]

        out_pos = (
            target_pos
            .loc[
                target_orig
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        local_times = (
            g[
                "_time_sec"
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        local_phase = (
            g[
                "_is_dep"
            ]
            .to_numpy(
                dtype=np.int8
            )
        )

        feats = _local_temporal_arrays(
            local_times,
            local_phase,
        )

        for k in airport_feature_keys:
            buffers[
                f"tg_airport_{k}"
            ][
                out_pos
            ] = feats[
                k
            ][
                local_target
            ]

        # Off-block -> takeoff interval counts at airport level.
        dep_cum = np.empty(
            len(
                local_times
            )
            + 1,
            dtype=np.int64,
        )

        arr_cum = np.empty(
            len(
                local_times
            )
            + 1,
            dtype=np.int64,
        )

        dep_cum[
            0
        ] = 0

        arr_cum[
            0
        ] = 0

        np.cumsum(
            local_phase,
            out=dep_cum[
                1:
            ],
        )

        np.cumsum(
            1
            - local_phase,
            out=arr_cum[
                1:
            ],
        )

        anchors = (
            target_anchor_sec
            .loc[
                target_orig
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        good = np.isfinite(
            anchors
        )

        if good.any():
            event_idx = local_target[
                good
            ]

            left = np.searchsorted(
                local_times,
                anchors[
                    good
                ],
                side="left",
            )

            # Bad/late NM proxies can fall after takeoff. Treat their interval
            # queue count as zero rather than creating negative counts.
            left = np.minimum(
                left,
                event_idx,
            )

            # Count events from anchor up to, but excluding, current takeoff.
            total_count = (
                event_idx
                - left
            )

            dep_count = (
                dep_cum[
                    event_idx
                ]
                - dep_cum[
                    left
                ]
            )

            arr_count = (
                arr_cum[
                    event_idx
                ]
                - arr_cum[
                    left
                ]
            )

            pos = out_pos[
                good
            ]

            buffers[
                "q_airport_movements_between"
            ][
                pos
            ] = total_count.astype(
                np.float32
            )

            buffers[
                "q_airport_departures_between"
            ][
                pos
            ] = dep_count.astype(
                np.float32
            )

            buffers[
                "q_airport_arrivals_between"
            ][
                pos
            ] = arr_count.astype(
                np.float32
            )

    print(
        "  runway temporal/queue graph..."
    )

    for _, g in valid_ctx.groupby(
        "_runway_group",
        sort=False,
    ):
        g = g.sort_values(
            "_time_sec",
            kind="mergesort",
        )

        orig = g[
            "_orig_index"
        ].to_numpy()

        keep_mask = (
            pd.Index(
                orig
            )
            .isin(
                dep_index
            )
        )

        if not keep_mask.any():
            continue

        local_target = np.flatnonzero(
            keep_mask
        )

        target_orig = orig[
            keep_mask
        ]

        out_pos = (
            target_pos
            .loc[
                target_orig
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        local_times = (
            g[
                "_time_sec"
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        local_phase = (
            g[
                "_is_dep"
            ]
            .to_numpy(
                dtype=np.int8
            )
        )

        feats = _local_temporal_arrays(
            local_times,
            local_phase,
        )

        for k in runway_feature_keys:
            buffers[
                f"tg_runway_{k}"
            ][
                out_pos
            ] = feats[
                k
            ][
                local_target
            ]

        dep_cum = np.empty(
            len(
                local_times
            )
            + 1,
            dtype=np.int64,
        )

        arr_cum = np.empty(
            len(
                local_times
            )
            + 1,
            dtype=np.int64,
        )

        dep_cum[
            0
        ] = 0

        arr_cum[
            0
        ] = 0

        np.cumsum(
            local_phase,
            out=dep_cum[
                1:
            ],
        )

        np.cumsum(
            1
            - local_phase,
            out=arr_cum[
                1:
            ],
        )

        anchors = (
            target_anchor_sec
            .loc[
                target_orig
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        good = np.isfinite(
            anchors
        )

        if good.any():
            event_idx = local_target[
                good
            ]

            left = np.searchsorted(
                local_times,
                anchors[
                    good
                ],
                side="left",
            )

            left = np.minimum(
                left,
                event_idx,
            )

            total_count = (
                event_idx
                - left
            )

            dep_count = (
                dep_cum[
                    event_idx
                ]
                - dep_cum[
                    left
                ]
            )

            arr_count = (
                arr_cum[
                    event_idx
                ]
                - arr_cum[
                    left
                ]
            )

            pos = out_pos[
                good
            ]

            buffers[
                "q_runway_movements_between"
            ][
                pos
            ] = total_count.astype(
                np.float32
            )

            buffers[
                "q_runway_departures_between"
            ][
                pos
            ] = dep_count.astype(
                np.float32
            )

            buffers[
                "q_runway_arrivals_between"
            ][
                pos
            ] = arr_count.astype(
                np.float32
            )

    result = departures.copy()

    feature_frame = pd.DataFrame(
        buffers,
        index=result.index,
    )

    result = pd.concat(
        [
            result,
            feature_frame,
        ],
        axis=1,
        copy=False,
    )

    eps = np.float32(
        1.0
    )

    for mins in [
        5,
        10,
        15,
    ]:
        result[
            f"tg_runway_share_{mins}m"
        ] = (
            result[
                f"tg_runway_deg_{mins}m"
            ]
            / (
                result[
                    f"tg_airport_deg_{mins}m"
                ]
                + eps
            )
        ).astype(
            "float32"
        )

        result[
            f"tg_dep_pressure_{mins}m"
        ] = (
            result[
                f"tg_airport_dep_{mins}m"
            ]
            - result[
                f"tg_airport_arr_{mins}m"
            ]
        ).astype(
            "float32"
        )

        result[
            f"tg_dep_fraction_{mins}m"
        ] = (
            result[
                f"tg_airport_dep_{mins}m"
            ]
            / (
                result[
                    f"tg_airport_dep_{mins}m"
                ]
                + result[
                    f"tg_airport_arr_{mins}m"
                ]
                + eps
            )
        ).astype(
            "float32"
        )

        result[
            f"tg_runway_dep_pressure_{mins}m"
        ] = (
            result[
                f"tg_runway_dep_{mins}m"
            ]
            - result[
                f"tg_runway_arr_{mins}m"
            ]
        ).astype(
            "float32"
        )

        result[
            f"tg_runway_dep_fraction_{mins}m"
        ] = (
            result[
                f"tg_runway_dep_{mins}m"
            ]
            / (
                result[
                    f"tg_runway_dep_{mins}m"
                ]
                + result[
                    f"tg_runway_arr_{mins}m"
                ]
                + eps
            )
        ).astype(
            "float32"
        )

    result[
        "tg_runway_vs_airport_exp300"
    ] = (
        result[
            "tg_runway_exp_degree_tau300"
        ]
        / (
            result[
                "tg_airport_exp_degree_tau300"
            ]
            + eps
        )
    ).astype(
        "float32"
    )

    result[
        "tg_runway_vs_airport_exp900"
    ] = (
        result[
            "tg_runway_exp_degree_tau900"
        ]
        / (
            result[
                "tg_airport_exp_degree_tau900"
            ]
            + eps
        )
    ).astype(
        "float32"
    )

    # Queue composition while aircraft is between selected NM off-block
    # timestamp and takeoff.
    result[
        "q_runway_fraction_between"
    ] = safe_divide(
        result[
            "q_runway_movements_between"
        ],
        result[
            "q_airport_movements_between"
        ],
    )

    result[
        "q_dep_fraction_between"
    ] = safe_divide(
        result[
            "q_airport_departures_between"
        ],
        result[
            "q_airport_movements_between"
        ],
    )

    result[
        "q_runway_dep_fraction_between"
    ] = safe_divide(
        result[
            "q_runway_departures_between"
        ],
        result[
            "q_runway_movements_between"
        ],
    )

    del (
        ctx,
        valid_ctx,
        buffers,
        feature_frame,
    )

    gc.collect()

    return result


print(
    "✓ V17 queue/temporal graph helpers defined."
)

In [ ]:
#@title 6. Departure datasets and integrity checks
TARGET = "TAXITIME_SEC_mvt"

train_dep = train_all[
    train_all["PHASE_mvt"].eq("DEP") &
    train_all[TARGET].notna()
].copy()

ranking_dep = ranking_all[
    ranking_all["PHASE_mvt"].eq("DEP")
].copy()

assert len(template) == len(ranking_dep)
assert set(template["MVT_ID_mvt"]) == set(ranking_dep["MVT_ID_mvt"])

print("Training departures:", f"{len(train_dep):,}")
print("Ranking departures :", f"{len(ranking_dep):,}")

display(
    train_dep[TARGET].describe(
        percentiles=[.001, .01, .05, .5, .95, .99, .999]
    )
)

In [ ]:
#@title 6b. Add dynamic graph features to departures only
print("Adding dynamic graph features to 2025 departures...")
train_dep = build_dynamic_features_for_departures(
    train_all,
    train_dep,
)

print("Adding dynamic graph features to ranking departures...")
ranking_dep = build_dynamic_features_for_departures(
    ranking_all,
    ranking_dep,
)

tg_cols = [c for c in train_dep.columns if c.startswith("tg_")]

print("✓ Dynamic graph features:", len(tg_cols))
print("Train departure shape:", train_dep.shape)
print("Ranking departure shape:", ranking_dep.shape)

# Release large full-movement temporary columns/data as soon as possible.
gc.collect()

# The full movement tables are no longer needed after temporal graph extraction.
# They are the largest retained objects at this stage.
report_ram("before releasing full movement tables")

del train_all
del ranking_all
gc.collect()

report_ram("after releasing full movement tables")


### Memory design

All 2025 movements are still used as temporal context, including arrivals, but the dynamic graph columns are stored only for departure rows.

This avoids expanding the ~4.17M-row full movement table with dozens of extra float columns, which can exceed Colab system RAM.

In [ ]:
#@title 7. Feature engineering
RAW_TIME_COLS = [
    "MVT_TIME_UTC_mvt",
    "SCHED_TIME_UTC_mvt",
    "AOBT_3_flt",
    "LOBT_flt",
    "IOBT_flt",
    "EOBT_1_flt",
    "ARVT_3_flt",
    "BLOCK_TIME_UTC_mvt",
]

DROP_ALWAYS = {
    TARGET,
    "BLOCK_TIME_UTC_mvt",
    "MVT_ID_mvt",
    "FLIGHT_ID_mvt",
    "FLIGHT_mvt",
    "CALLSIGN_flt",
    "PHASE_mvt",
    "ARVT_3_flt",
    # NM timestamps become engineered features, not raw datetime columns.
    "AOBT_3_flt",
    "LOBT_flt",
    "IOBT_flt",
    "EOBT_1_flt",
    "MVT_TIME_UTC_mvt",
    "SCHED_TIME_UTC_mvt",
}

def add_calendar_and_nm_features(df):
    out = df.copy()

    sched = to_utc(out["SCHED_TIME_UTC_mvt"])
    mvt = to_utc(out["MVT_TIME_UTC_mvt"])

    # Schedule calendar.
    out["sched_hour"] = sched.dt.hour.astype("float32")
    out["sched_dow"] = sched.dt.dayofweek.astype("float32")
    out["sched_month"] = sched.dt.month.astype("float32")
    out["sched_day"] = sched.dt.day.astype("float32")
    out["sched_week"] = sched.dt.isocalendar().week.astype("float32")
    out["sched_weekend"] = (sched.dt.dayofweek >= 5).astype("int8")

    out["sched_hour_sin"] = np.sin(2*np.pi*out["sched_hour"]/24)
    out["sched_hour_cos"] = np.cos(2*np.pi*out["sched_hour"]/24)
    out["sched_dow_sin"] = np.sin(2*np.pi*out["sched_dow"]/7)
    out["sched_dow_cos"] = np.cos(2*np.pi*out["sched_dow"]/7)
    out["sched_month_sin"] = np.sin(2*np.pi*out["sched_month"]/12)
    out["sched_month_cos"] = np.cos(2*np.pi*out["sched_month"]/12)

    # Actual movement calendar, useful for retrospective reconstruction.
    out["mvt_hour"] = mvt.dt.hour.astype("float32")
    out["mvt_dow"] = mvt.dt.dayofweek.astype("float32")
    out["mvt_month"] = mvt.dt.month.astype("float32")

    parsed = {}
    for c in ["AOBT_3_flt", "LOBT_flt", "IOBT_flt", "EOBT_1_flt"]:
        if c in out.columns:
            parsed[c] = to_utc(out[c])

    def delta_min(a, b_series, name):
        if a in parsed:
            out[name] = (parsed[a] - b_series).dt.total_seconds() / 60.0

    delta_min("LOBT_flt", sched, "lobt_minus_sched_min")
    delta_min("IOBT_flt", sched, "iobt_minus_sched_min")
    delta_min("EOBT_1_flt", sched, "eobt_minus_sched_min")
    delta_min("AOBT_3_flt", sched, "aobt_minus_sched_min")

    if "LOBT_flt" in parsed and "IOBT_flt" in parsed:
        out["lobt_minus_iobt_min"] = (
            parsed["LOBT_flt"] - parsed["IOBT_flt"]
        ).dt.total_seconds() / 60.0

    if "EOBT_1_flt" in parsed and "IOBT_flt" in parsed:
        out["eobt_minus_iobt_min"] = (
            parsed["EOBT_1_flt"] - parsed["IOBT_flt"]
        ).dt.total_seconds() / 60.0

    return out

def add_nm_anchor(df):
    out = df.copy()

    mvt = to_utc(out["MVT_TIME_UTC_mvt"])

    proxy_order = [
        ("AOBT_3_flt", "AOBT"),
        ("EOBT_1_flt", "EOBT"),
        ("IOBT_flt", "IOBT"),
        ("LOBT_flt", "LOBT"),
    ]

    selected = pd.Series(pd.NaT, index=out.index, dtype="datetime64[ns, UTC]")
    source = pd.Series("__MISSING__", index=out.index, dtype="object")

    for col, label in proxy_order:
        if col not in out.columns:
            continue
        candidate = to_utc(out[col])
        mask = selected.isna() & candidate.notna()
        selected.loc[mask] = candidate.loc[mask]
        source.loc[mask] = label

    anchor_sec = (mvt - selected).dt.total_seconds()

    out["nm_anchor_sec"] = anchor_sec.astype("float32")
    out["nm_anchor_source"] = source.astype(str)
    out["nm_anchor_missing"] = anchor_sec.isna().astype("int8")

    return out

def make_X(df, feature_columns=None, cat_columns=None, include_anchor=False):
    X = add_calendar_and_nm_features(df)

    if include_anchor:
        X = add_nm_anchor(X)

    drop = [c for c in DROP_ALWAYS if c in X.columns]
    X = X.drop(columns=drop, errors="ignore")

    dt_cols = [
        c for c in X.columns
        if pd.api.types.is_datetime64_any_dtype(X[c])
        or pd.api.types.is_timedelta64_dtype(X[c])
    ]
    X = X.drop(columns=dt_cols, errors="ignore")

    id_like = [
        c for c in X.columns
        if c.upper().endswith("_ID")
        or c.upper().startswith("ID_")
    ]
    X = X.drop(columns=id_like, errors="ignore")

    if feature_columns is None:
        feature_columns = list(X.columns)

    X = X.reindex(columns=feature_columns)

    if cat_columns is None:
        cat_columns = []
        for c in X.columns:
            dtype = X[c].dtype
            if not (
                pd.api.types.is_numeric_dtype(dtype)
                or pd.api.types.is_bool_dtype(dtype)
            ):
                cat_columns.append(c)

    for c in cat_columns:
        if c in X.columns:
            X[c] = X[c].astype("string").fillna("__MISSING__").astype(str)

    for c in X.columns:
        if c in cat_columns:
            continue
        if pd.api.types.is_bool_dtype(X[c].dtype):
            X[c] = X[c].astype("int8")
        else:
            X[c] = pd.to_numeric(X[c], errors="coerce")

    bad = [
        c for c in X.columns
        if c not in cat_columns
        and not pd.api.types.is_numeric_dtype(X[c].dtype)
    ]
    if bad:
        raise TypeError(f"Non-numeric columns outside cat_features: {bad}")

    return X, feature_columns, cat_columns


def audit_feature_types(X, cat_columns, title="feature matrix"):
    numeric = [c for c in X.columns if c not in cat_columns]
    print(f"{title}: {X.shape[0]:,} rows × {X.shape[1]} features")
    print(f"  categorical: {len(cat_columns)}")
    print(f"  numeric:     {len(numeric)}")

# ============================================================
# V7.2 memory-safe overrides
# ============================================================

def _downcast_numeric_inplace(df, exclude=None):
    """
    Downcast numeric columns in-place without copying the full DataFrame.
    """
    if exclude is None:
        exclude = set()
    else:
        exclude = set(exclude)

    for c in df.columns:
        if c in exclude:
            continue

        dtype = df[c].dtype

        if pd.api.types.is_float_dtype(dtype):
            if dtype != np.float32:
                df[c] = pd.to_numeric(
                    df[c],
                    errors="coerce",
                    downcast="float",
                ).astype("float32")

        elif pd.api.types.is_integer_dtype(dtype):
            # int32 is plenty for engineered counts / flags.
            if dtype.itemsize > 4:
                df[c] = pd.to_numeric(
                    df[c],
                    errors="coerce",
                    downcast="integer",
                )

    return df


def add_calendar_and_nm_features_memory_safe(df):
    """
    Memory-safe version:
    create only the feature frame LightGBM needs instead of copying
    the entire input DataFrame first.
    """
    # Start with a shallow frame. Individual new columns are materialized below.
    out = df.copy(deep=False)

    # Reuse the existing calendar/NM feature function if available,
    # but only on a narrow base frame where possible.
    #
    # We construct derived columns explicitly to avoid a full deep copy.
    sched = to_utc(out["SCHED_TIME_UTC_mvt"])

    out["sched_hour"] = sched.dt.hour.astype("float32")
    out["sched_dow"] = sched.dt.dayofweek.astype("float32")
    out["sched_month"] = sched.dt.month.astype("float32")
    out["sched_day"] = sched.dt.day.astype("float32")
    out["sched_week"] = sched.dt.isocalendar().week.astype("float32")

    out["sched_hour_sin"] = np.sin(
        2 * np.pi * out["sched_hour"] / 24.0
    ).astype("float32")
    out["sched_hour_cos"] = np.cos(
        2 * np.pi * out["sched_hour"] / 24.0
    ).astype("float32")

    out["sched_dow_sin"] = np.sin(
        2 * np.pi * out["sched_dow"] / 7.0
    ).astype("float32")
    out["sched_dow_cos"] = np.cos(
        2 * np.pi * out["sched_dow"] / 7.0
    ).astype("float32")

    # Preserve NM anchor-derived features if they already exist.
    # add_nm_anchor is shallow enough here because we are not doing a deep copy.
    out = add_nm_anchor(out)

    return out


def make_X_memory_safe(
    df,
    feature_columns=None,
    cat_columns=None,
    include_anchor=True,
):
    """
    LightGBM feature matrix builder designed to avoid repeated deep copies.
    """
    work = add_calendar_and_nm_features_memory_safe(df)

    # Keep graph / target-prior / traffic / engineered numeric columns already present.
    # Remove target and obvious leakage/output identifiers.
    drop_cols = {
        TARGET,
        "BLOCK_TIME_UTC_mvt",
    }

    # Preserve the original make_X feature-selection behavior by consulting
    # existing explicit feature lists when feature_columns is provided.
    if feature_columns is None:
        candidate_cols = [
            c for c in work.columns
            if c not in drop_cols
        ]

        # Do not feed raw datetime columns directly to CatBoost.
        datetime_cols = [
            c for c in candidate_cols
            if pd.api.types.is_datetime64_any_dtype(work[c])
            or pd.api.types.is_datetime64tz_dtype(work[c])
        ]
        candidate_cols = [
            c for c in candidate_cols
            if c not in datetime_cols
        ]

        # Remove obvious IDs / free-text columns that the original pipeline
        # should not model directly.
        blocked = {
            "MVT_ID_mvt",
        }
        candidate_cols = [
            c for c in candidate_cols
            if c not in blocked
        ]

        if not include_anchor:
            candidate_cols = [
                c for c in candidate_cols
                if c != "nm_anchor_sec"
            ]

        feature_columns = candidate_cols

    # Build only selected columns.
    X = work.loc[:, feature_columns].copy()

    if cat_columns is None:
        cat_columns = [
            c for c in X.columns
            if (
                pd.api.types.is_object_dtype(X[c])
                or pd.api.types.is_string_dtype(X[c])
                or str(X[c].dtype) == "category"
            )
        ]

    # categorical columns: stable strings, fill missing.
    for c in cat_columns:
        X[c] = X[c].astype("string").fillna("__MISSING__").astype(str)

    # Aggressive numeric downcast after categorical conversion.
    numeric_cols = [c for c in X.columns if c not in cat_columns]

    for c in numeric_cols:
        if pd.api.types.is_bool_dtype(X[c]):
            X[c] = X[c].astype("int8")
        else:
            X[c] = pd.to_numeric(
                X[c],
                errors="coerce",
                downcast="float",
            ).astype("float32")

    return X, list(feature_columns), list(cat_columns)


def prepare_X_inplace(
    df,
    feature_columns=None,
    cat_columns=None,
    include_anchor=True,
):
    """
    V7.3 low-RAM matrix preparation.

    Mutates the supplied DataFrame in place and then returns that SAME object
    as X. No second full feature matrix is created.
    """
    # Calendar features directly on this frame.
    sched = to_utc(df["SCHED_TIME_UTC_mvt"])

    df["sched_hour"] = sched.dt.hour.astype("float32")
    df["sched_dow"] = sched.dt.dayofweek.astype("float32")
    df["sched_month"] = sched.dt.month.astype("float32")
    df["sched_day"] = sched.dt.day.astype("float32")
    df["sched_week"] = sched.dt.isocalendar().week.astype("float32")

    df["sched_hour_sin"] = np.sin(
        2 * np.pi * df["sched_hour"] / 24.0
    ).astype("float32")
    df["sched_hour_cos"] = np.cos(
        2 * np.pi * df["sched_hour"] / 24.0
    ).astype("float32")
    df["sched_dow_sin"] = np.sin(
        2 * np.pi * df["sched_dow"] / 7.0
    ).astype("float32")
    df["sched_dow_cos"] = np.cos(
        2 * np.pi * df["sched_dow"] / 7.0
    ).astype("float32")

    # Ensure anchor exists without creating another full deep frame.
    if "nm_anchor_sec" not in df.columns:
        tmp = add_nm_anchor(df)
        new_cols = [
            c for c in tmp.columns
            if c not in df.columns
        ]
        for c in new_cols:
            df[c] = tmp[c].to_numpy(copy=False)
        del tmp
        gc.collect()

    if feature_columns is None:
        blocked = {
            TARGET,
            "BLOCK_TIME_UTC_mvt",
            "MVT_ID_mvt",
        }

        feature_columns = []

        for c in df.columns:
            if c in blocked:
                continue

            if (
                pd.api.types.is_datetime64_any_dtype(df[c])
                or pd.api.types.is_datetime64tz_dtype(df[c])
            ):
                continue

            if (not include_anchor) and c == "nm_anchor_sec":
                continue

            feature_columns.append(c)

    # Determine categoricals before dropping columns.
    if cat_columns is None:
        cat_columns = [
            c for c in feature_columns
            if (
                pd.api.types.is_object_dtype(df[c])
                or pd.api.types.is_string_dtype(df[c])
                or str(df[c].dtype) == "category"
            )
        ]

    # Drop everything the model will not consume.
    keep = set(feature_columns)
    drop_now = [
        c for c in df.columns
        if c not in keep
    ]

    if drop_now:
        df.drop(columns=drop_now, inplace=True)

    gc.collect()

    # Convert columns in place.
    for c in feature_columns:
        if c in cat_columns:
            df[c] = (
                df[c]
                .astype("string")
                .fillna("__MISSING__")
                .astype(str)
            )
        else:
            if pd.api.types.is_bool_dtype(df[c]):
                df[c] = df[c].astype("int8")
            else:
                df[c] = pd.to_numeric(
                    df[c],
                    errors="coerce",
                    downcast="float",
                ).astype("float32")

    # Preserve exact feature order without .loc copy.
    # Reindexing columns may create a manager-level copy, so instead only
    # return the current frame and its actual order.
    actual_features = list(df.columns)

    return df, actual_features, list(cat_columns)


# ============================================================
# V7.5: EXACT ORIGINAL METHOD-B FEATURE LOGIC + GRAPH ONLY
# ============================================================

ORIGINAL_DROP_ALWAYS = {
    TARGET,
    "BLOCK_TIME_UTC_mvt",
    "MVT_ID_mvt",
    "FLIGHT_ID_mvt",
    "FLIGHT_mvt",
    "CALLSIGN_flt",
    "PHASE_mvt",
    "ARVT_3_flt",
    "AOBT_3_flt",
    "LOBT_flt",
    "IOBT_flt",
    "EOBT_1_flt",
    "MVT_TIME_UTC_mvt",
    "SCHED_TIME_UTC_mvt",
}

# These V6/V7 experimental columns are deliberately NOT CatBoost inputs.
V75_EXPERIMENTAL_EXCLUDE = {
    "baseline_sec",
    "baseline_is_nm",
    "baseline_is_prior",
    "nm_minus_prior_sec",
}


def _v17_string_series(
    df,
    col,
):
    if col not in df.columns:
        return pd.Series(
            "__MISSING__",
            index=df.index,
            dtype="string",
        )

    return (
        df[
            col
        ]
        .astype(
            "string"
        )
        .fillna(
            "__MISSING__"
        )
    )


def _v17_match_flag(
    df,
    left_col,
    right_col,
    output_col,
):
    """
    1 = both available and equal
    0 = both available and different
    NaN = at least one side unavailable
    """
    if (
        left_col not in df.columns
        or right_col not in df.columns
    ):
        return

    left = (
        df[
            left_col
        ]
        .astype(
            "string"
        )
    )

    right = (
        df[
            right_col
        ]
        .astype(
            "string"
        )
    )

    both = (
        left.notna()
        & right.notna()
    )

    out = np.full(
        len(
            df
        ),
        np.nan,
        dtype=np.float32,
    )

    if both.any():
        out[
            both.to_numpy()
        ] = (
            left.loc[
                both
            ]
            .eq(
                right.loc[
                    both
                ]
            )
            .astype(
                np.float32
            )
            .to_numpy()
        )

    df[
        output_col
    ] = out


def add_v17_operational_features_inplace(
    df,
):
    """
    High-value operational features built before raw flight/callsign columns
    are dropped.

    Composite identities are stored as pandas categoricals to keep RAM small.
    """
    airport = _v17_string_series(
        df,
        "ADEP_mvt",
    )

    runway = _v17_string_series(
        df,
        "RUNWAY_mvt",
    )

    stand = _v17_string_series(
        df,
        "STAND_mvt",
    )

    destination = _v17_string_series(
        df,
        "ADES_mvt",
    )

    aircraft = _v17_string_series(
        df,
        "AIRCRAFT_TYPE_mvt",
    )

    if (
        "AIRCRAFT_OPERATOR_flt"
        in df.columns
    ):
        operator = _v17_string_series(
            df,
            "AIRCRAFT_OPERATOR_flt",
        )

    elif (
        "OPERATOR_flt"
        in df.columns
    ):
        operator = _v17_string_series(
            df,
            "OPERATOR_flt",
        )

    else:
        callsign = _v17_string_series(
            df,
            "CALLSIGN_flt",
        )

        operator = (
            callsign
            .str.extract(
                r"^([A-Za-z]+)",
                expand=False,
            )
            .fillna(
                "__MISSING__"
            )
            .astype(
                "string"
            )
        )

    # Flight number / recurrence identity.
    if (
        "FLIGHT_mvt"
        in df.columns
    ):
        flight_raw = _v17_string_series(
            df,
            "FLIGHT_mvt",
        )

    else:
        flight_raw = _v17_string_series(
            df,
            "CALLSIGN_flt",
        )

    flight_number = (
        flight_raw
        .str.extract(
            r"(\d+)",
            expand=False,
        )
        .fillna(
            "__MISSING__"
        )
        .astype(
            "string"
        )
    )

    # Compact categorical interactions.
    composite_specs = {
        "cx_airport_runway": (
            airport
            + "|"
            + runway
        ),
        "cx_airport_stand": (
            airport
            + "|"
            + stand
        ),
        "cx_airport_stand_runway": (
            airport
            + "|"
            + stand
            + "|"
            + runway
        ),
        "cx_operator_route": (
            operator
            + "|"
            + airport
            + "|"
            + destination
        ),
        "cx_operator_runway": (
            operator
            + "|"
            + airport
            + "|"
            + runway
        ),
        "cx_operator_flight": (
            operator
            + "|"
            + flight_number
        ),
        "cx_aircraft_runway": (
            aircraft
            + "|"
            + airport
            + "|"
            + runway
        ),
    }

    for name, values in composite_specs.items():
        df[
            name
        ] = values.astype(
            "category"
        )

    # Keep decomposed operator/flight recurrence as explicit categories too.
    df[
        "cx_operator_code"
    ] = operator.astype(
        "category"
    )

    df[
        "cx_flight_number"
    ] = flight_number.astype(
        "category"
    )

    # NM / movement-record reliability indicators.
    _v17_match_flag(
        df,
        "ADEP_mvt",
        "ADEP_flt",
        "nm_match_adep",
    )

    _v17_match_flag(
        df,
        "ADES_mvt",
        "ADES_flt",
        "nm_match_ades",
    )

    _v17_match_flag(
        df,
        "AIRCRAFT_TYPE_mvt",
        "AIRCRAFT_TYPE_flt",
        "nm_match_aircraft_type",
    )

    _v17_match_flag(
        df,
        "REGISTRATION_mvt",
        "REGISTRATION_flt",
        "nm_match_registration",
    )

    match_cols = [
        c
        for c in [
            "nm_match_adep",
            "nm_match_ades",
            "nm_match_aircraft_type",
            "nm_match_registration",
        ]
        if c in df.columns
    ]

    if match_cols:
        mat = (
            df[
                match_cols
            ]
            .to_numpy(
                dtype=np.float32,
                copy=False,
            )
        )

        valid = np.isfinite(
            mat
        )

        count = valid.sum(
            axis=1
        ).astype(
            np.float32
        )

        score_sum = np.nansum(
            mat,
            axis=1,
        ).astype(
            np.float32
        )

        df[
            "nm_match_fields_available"
        ] = count

        df[
            "nm_match_quality"
        ] = np.divide(
            score_sum,
            count,
            out=np.full(
                len(
                    df
                ),
                np.nan,
                dtype=np.float32,
            ),
            where=count > 0,
        )

        df[
            "nm_match_mismatch_count"
        ] = (
            count
            - score_sum
        ).astype(
            np.float32
        )

    # Schedule slot is useful for recurring operations without requiring
    # LightGBM to reconstruct 15-minute bins from a continuous hour.
    if (
        "SCHED_TIME_UTC_mvt"
        in df.columns
    ):
        sched = to_utc(
            df[
                "SCHED_TIME_UTC_mvt"
            ]
        )

        df[
            "sched_slot_15m"
        ] = (
            (
                sched.dt.hour
                * 4
            )
            + (
                sched.dt.minute
                // 15
            )
        ).astype(
            "float32"
        )

        slot_str = (
            df[
                "sched_slot_15m"
            ]
            .fillna(
                -1
            )
            .astype(
                "int16"
            )
            .astype(
                "string"
            )
        )

        df[
            "cx_operator_route_slot"
        ] = (
            operator
            + "|"
            + airport
            + "|"
            + destination
            + "|"
            + slot_str
        ).astype(
            "category"
        )

    return df


def add_original_method_b_derived_features_inplace(df):
    """
    Reproduce the original V2 add_calendar_and_nm_features + add_nm_anchor
    logic without a full deep DataFrame copy.
    """
    add_v17_operational_features_inplace(
        df
    )

    sched = to_utc(df["SCHED_TIME_UTC_mvt"])
    mvt = to_utc(df["MVT_TIME_UTC_mvt"])

    # Original schedule calendar features.
    df["sched_hour"] = sched.dt.hour.astype("float32")
    df["sched_dow"] = sched.dt.dayofweek.astype("float32")
    df["sched_month"] = sched.dt.month.astype("float32")
    df["sched_day"] = sched.dt.day.astype("float32")
    df["sched_week"] = sched.dt.isocalendar().week.astype("float32")
    df["sched_weekend"] = (
        sched.dt.dayofweek >= 5
    ).astype("int8")

    df["sched_hour_sin"] = np.sin(
        2 * np.pi * df["sched_hour"] / 24
    ).astype("float32")

    df["sched_hour_cos"] = np.cos(
        2 * np.pi * df["sched_hour"] / 24
    ).astype("float32")

    df["sched_dow_sin"] = np.sin(
        2 * np.pi * df["sched_dow"] / 7
    ).astype("float32")

    df["sched_dow_cos"] = np.cos(
        2 * np.pi * df["sched_dow"] / 7
    ).astype("float32")

    df["sched_month_sin"] = np.sin(
        2 * np.pi * df["sched_month"] / 12
    ).astype("float32")

    df["sched_month_cos"] = np.cos(
        2 * np.pi * df["sched_month"] / 12
    ).astype("float32")

    # Original retrospective movement calendar.
    df["mvt_hour"] = mvt.dt.hour.astype("float32")
    df["mvt_dow"] = mvt.dt.dayofweek.astype("float32")
    df["mvt_month"] = mvt.dt.month.astype("float32")

    parsed = {}

    for c in [
        "AOBT_3_flt",
        "LOBT_flt",
        "IOBT_flt",
        "EOBT_1_flt",
    ]:
        if c in df.columns:
            parsed[c] = to_utc(df[c])

    def delta_min(col, reference, name):
        if col in parsed:
            df[name] = (
                (parsed[col] - reference)
                .dt.total_seconds()
                .div(60.0)
                .astype("float32")
            )

    delta_min(
        "LOBT_flt",
        sched,
        "lobt_minus_sched_min",
    )

    delta_min(
        "IOBT_flt",
        sched,
        "iobt_minus_sched_min",
    )

    delta_min(
        "EOBT_1_flt",
        sched,
        "eobt_minus_sched_min",
    )

    delta_min(
        "AOBT_3_flt",
        sched,
        "aobt_minus_sched_min",
    )

    if (
        "LOBT_flt" in parsed
        and "IOBT_flt" in parsed
    ):
        df["lobt_minus_iobt_min"] = (
            (parsed["LOBT_flt"] - parsed["IOBT_flt"])
            .dt.total_seconds()
            .div(60.0)
            .astype("float32")
        )

    if (
        "EOBT_1_flt" in parsed
        and "IOBT_flt" in parsed
    ):
        df["eobt_minus_iobt_min"] = (
            (parsed["EOBT_1_flt"] - parsed["IOBT_flt"])
            .dt.total_seconds()
            .div(60.0)
            .astype("float32")
        )

    # Original NM anchor logic.
    proxy_order = [
        ("AOBT_3_flt", "AOBT"),
        ("EOBT_1_flt", "EOBT"),
        ("IOBT_flt", "IOBT"),
        ("LOBT_flt", "LOBT"),
    ]

    selected = pd.Series(
        pd.NaT,
        index=df.index,
        dtype="datetime64[ns, UTC]",
    )

    source = pd.Series(
        "__MISSING__",
        index=df.index,
        dtype="object",
    )

    for col, label in proxy_order:
        if col not in df.columns:
            continue

        candidate = to_utc(df[col])
        mask = (
            selected.isna()
            & candidate.notna()
        )

        selected.loc[mask] = candidate.loc[mask]
        source.loc[mask] = label

    anchor_sec = (
        mvt - selected
    ).dt.total_seconds()

    df["nm_anchor_sec"] = anchor_sec.astype("float32")
    df["nm_anchor_source"] = source.astype(str)
    df["nm_anchor_missing"] = anchor_sec.isna().astype("int8")

    return df


def make_original_method_b_plus_graph_inplace(
    df,
    feature_columns=None,
    cat_columns=None,
):
    """
    Exact original Method-B selection logic, with only g_* and tg_*
    added by virtue of already existing in df.

    prior_* and V6/V7 baseline features are excluded; V17 operational features are retained.
    """
    add_original_method_b_derived_features_inplace(df)

    # Original drop behavior.
    drop_cols = [
        c
        for c in ORIGINAL_DROP_ALWAYS
        if c in df.columns
    ]

    # Clean ablation: priors are fallback-only, not model inputs.
    drop_cols += [
        c
        for c in df.columns
        if c.startswith("prior_")
    ]

    drop_cols += [
        c
        for c in V75_EXPERIMENTAL_EXCLUDE
        if c in df.columns
    ]

    if drop_cols:
        df.drop(
            columns=list(dict.fromkeys(drop_cols)),
            inplace=True,
            errors="ignore",
        )

    # Original datetime / timedelta removal.
    dt_cols = [
        c
        for c in df.columns
        if (
            pd.api.types.is_datetime64_any_dtype(df[c])
            or pd.api.types.is_timedelta64_dtype(df[c])
        )
    ]

    if dt_cols:
        df.drop(
            columns=dt_cols,
            inplace=True,
            errors="ignore",
        )

    # Original generic ID-like removal.
    id_like = [
        c
        for c in df.columns
        if (
            c.upper().endswith("_ID")
            or c.upper().startswith("ID_")
        )
    ]

    if id_like:
        df.drop(
            columns=id_like,
            inplace=True,
            errors="ignore",
        )

    if feature_columns is None:
        feature_columns = list(df.columns)
    else:
        # Drop any extra columns not in frozen feature list.
        extra = [
            c
            for c in df.columns
            if c not in feature_columns
        ]
        if extra:
            df.drop(
                columns=extra,
                inplace=True,
                errors="ignore",
            )

        missing = [
            c
            for c in feature_columns
            if c not in df.columns
        ]

        if missing:
            for c in missing:
                df[c] = np.nan

        # DataFrame is small enough at validation time to reorder if needed.
        if list(df.columns) != list(feature_columns):
            df = df.reindex(columns=feature_columns)

    if cat_columns is None:
        cat_columns = []

        for c in feature_columns:
            dtype = df[c].dtype

            if not (
                pd.api.types.is_numeric_dtype(dtype)
                or pd.api.types.is_bool_dtype(dtype)
            ):
                cat_columns.append(c)

    # Original categorical coercion.
    for c in cat_columns:
        if c in df.columns:
            df[c] = (
                df[c]
                .astype("string")
                .fillna("__MISSING__")
                .astype(str)
            )

    # Original numeric coercion, with float32 downcast for RAM.
    for c in feature_columns:
        if c in cat_columns:
            continue

        if pd.api.types.is_bool_dtype(df[c].dtype):
            df[c] = df[c].astype("int8")
        else:
            df[c] = pd.to_numeric(
                df[c],
                errors="coerce",
            )

            if pd.api.types.is_float_dtype(df[c]):
                df[c] = df[c].astype("float32")

    bad = [
        c
        for c in feature_columns
        if (
            c not in cat_columns
            and not pd.api.types.is_numeric_dtype(df[c].dtype)
        )
    ]

    if bad:
        raise TypeError(
            f"Non-numeric columns outside cat_features: {bad}"
        )

    return df, list(feature_columns), list(cat_columns)


In [ ]:
#@title 7b. Historical prior helper functions
PRIOR_SPECS = [
    (["ADEP_mvt"], "prior_airport"),
    (["ADEP_mvt", "RUNWAY_mvt"], "prior_airport_runway"),
    (["ADEP_mvt", "_sched_hour"], "prior_airport_hour"),
    (["ADEP_mvt", "RUNWAY_mvt", "_sched_hour"], "prior_airport_runway_hour"),
    (["ADEP_mvt", "STAND_mvt"], "prior_airport_stand"),
]

def add_sched_hour(df):
    out = df.copy()
    out["_sched_hour"] = to_utc(out["SCHED_TIME_UTC_mvt"]).dt.hour
    return out

def fit_target_priors(train_df):
    hist = add_sched_hour(train_df)
    global_median = float(hist[TARGET].median())
    tables = []

    for keys, name in PRIOR_SPECS:
        stats = (
            hist.groupby(keys, dropna=False)[TARGET]
            .agg(["median", "count"])
            .reset_index()
        )

        # Stability threshold: more specific combinations require fewer rows.
        min_count = 20 if len(keys) == 1 else 12
        stats = stats.loc[stats["count"] >= min_count, keys + ["median"]]
        stats = stats.rename(columns={"median": name})
        tables.append((keys, name, stats))

    return {
        "global_median": global_median,
        "tables": tables,
    }

def apply_target_priors(df, prior_state):
    out = add_sched_hour(df)

    for keys, name, stats in prior_state["tables"]:
        tmp = (
            out[keys]
            .reset_index()
            .merge(stats, on=keys, how="left")
            .set_index("index")[name]
        )
        out[name] = tmp.reindex(out.index)

    # Backoff chain for missing prior values.
    out["prior_airport"] = out["prior_airport"].fillna(prior_state["global_median"])
    out["prior_airport_runway"] = out["prior_airport_runway"].fillna(out["prior_airport"])
    out["prior_airport_hour"] = out["prior_airport_hour"].fillna(out["prior_airport"])
    out["prior_airport_runway_hour"] = out["prior_airport_runway_hour"].fillna(
        out["prior_airport_runway"]
    ).fillna(out["prior_airport_hour"])
    out["prior_airport_stand"] = out["prior_airport_stand"].fillna(out["prior_airport"])

    # Differences between local priors are useful bias features.
    out["prior_runway_delta"] = out["prior_airport_runway"] - out["prior_airport"]
    out["prior_hour_delta"] = out["prior_airport_hour"] - out["prior_airport"]
    out["prior_rw_hour_delta"] = (
        out["prior_airport_runway_hour"] - out["prior_airport"]
    )
    out["prior_stand_delta"] = out["prior_airport_stand"] - out["prior_airport"]

    return out

## Missing-anchor deterministic fallback

The ~1% missing-anchor rows are handled without a second ML model.

V7.6 fits historical statistics specifically on **training rows that also lack an NM anchor**, then compares them against the general historical prior on October.

### V7.6.1 fix
Hierarchy join keys are canonicalized to strings before both grouping and merging, so missing-heavy categorical columns cannot produce pandas dtype mismatch errors.


In [ ]:
#@title 7bb. Missing-anchor fallback helpers — dtype-safe

MISSING_HIERARCHIES = [
    ["ADEP_mvt", "RUNWAY_mvt", "_sched_hour"],
    ["ADEP_mvt", "_sched_hour"],
    ["ADEP_mvt", "RUNWAY_mvt"],
    ["ADEP_mvt", "AIRCRAFT_TYPE_mvt"],
    ["ADEP_mvt", "AIRCRAFT_OPERATOR_flt"],
    ["ADEP_mvt"],
    ["_sched_hour"],
]

def _add_missing_sched_hour(df):
    out = df.copy(deep=False)
    out["_sched_hour"] = to_utc(
        out["SCHED_TIME_UTC_mvt"]
    ).dt.hour
    return out

def _canonicalize_hierarchy_keys(df, keys):
    """
    Force hierarchy merge/group keys to a stable dtype.

    This prevents pandas merge errors such as:
      object vs float64 for AIRCRAFT_OPERATOR_flt

    All hierarchy keys are used only for exact matching, so string
    canonicalization is safe here.
    """
    out = df.copy(deep=False)

    for k in keys:
        if k not in out.columns:
            continue

        s = out[k]

        # Convert missing values to a stable sentinel and everything else to str.
        out[k] = (
            s.astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

    return out

def fit_missing_fallback_state(train_df):
    tmp = add_nm_anchor(train_df)

    hist = tmp.loc[
        tmp["nm_anchor_sec"].isna()
    ].copy()

    hist = _add_missing_sched_hour(hist)

    if len(hist) == 0:
        raise ValueError(
            "No missing-anchor training rows."
        )

    tables = []

    for keys in MISSING_HIERARCHIES:
        keys = [
            k for k in keys
            if k in hist.columns
        ]

        if not keys:
            continue

        hist_keys = _canonicalize_hierarchy_keys(
            hist,
            keys,
        )

        stats = (
            hist_keys.groupby(
                keys,
                dropna=False,
            )[TARGET]
            .agg(
                mean="mean",
                median="median",
                count="size",
            )
            .reset_index()
        )

        min_count = (
            6 if len(keys) >= 3
            else 10 if len(keys) == 2
            else 20
        )

        stats = stats.loc[
            stats["count"] >= min_count
        ].copy()

        tables.append({
            "keys": keys,
            "stats": stats,
        })

    return {
        "global_mean": float(
            hist[TARGET].mean()
        ),
        "global_median": float(
            hist[TARGET].median()
        ),
        "tables": tables,
        "n_missing_train": int(
            len(hist)
        ),
    }

def predict_missing_hierarchy(
    df,
    state,
    statistic="median",
):
    work = _add_missing_sched_hour(df)

    pred = pd.Series(
        np.nan,
        index=work.index,
        dtype="float64",
    )

    for item in state["tables"]:
        keys = item["keys"]
        stats = item["stats"].copy()

        # Canonicalize BOTH sides with the exact same rule.
        work_keys = _canonicalize_hierarchy_keys(
            work,
            keys,
        )
        stats = _canonicalize_hierarchy_keys(
            stats,
            keys,
        )

        lookup = stats[
            keys + [statistic]
        ]

        probe = (
            work_keys[keys]
            .reset_index()
            .merge(
                lookup,
                on=keys,
                how="left",
                validate="many_to_one",
            )
            .set_index("index")[statistic]
        )

        mask = (
            pred.isna()
            & probe.notna()
        )

        pred.loc[mask] = probe.loc[mask]

    fallback = (
        state["global_mean"]
        if statistic == "mean"
        else state["global_median"]
    )

    return pred.fillna(
        fallback
    ).to_numpy(
        dtype=np.float64
    )

In [ ]:
#@title 7c. Static graph embedding helpers — challenge columns only
from scipy import sparse
from sklearn.decomposition import TruncatedSVD

def _clean_graph_value(v):
    if pd.isna(v):
        return "__MISSING__"
    return str(v)

def _graph_node_key(kind, value, airport=None):
    value = _clean_graph_value(value)

    # Runway labels can repeat across airports, so namespace them by airport.
    if kind == "runway":
        airport = _clean_graph_value(airport)
        return f"runway:{airport}|{value}"

    return f"{kind}:{value}"


def build_graph_embedding_state(
    df,
    dim=8,
    min_edge_count=3,
    random_state=42,
):
    """
    Static heterogeneous graph using only columns known to exist:
        airport <-> runway
        airport <-> destination
        runway  <-> destination

    No target values are used.
    """
    cols = ["ADEP_mvt", "ADES_mvt", "RUNWAY_mvt"]
    work = df.loc[:, [c for c in cols if c in df.columns]]

    edge_counts = {}

    def add_edge(a, b):
        if a == b:
            return
        if a > b:
            a, b = b, a
        edge_counts[(a, b)] = edge_counts.get((a, b), 0) + 1

    # itertuples is memory-light; no full object copy.
    for row in work.itertuples(index=False, name=None):
        d = dict(zip(work.columns, row))

        airport = _graph_node_key(
            "airport",
            d.get("ADEP_mvt"),
        )

        destination = _graph_node_key(
            "destination",
            d.get("ADES_mvt"),
        )

        runway = _graph_node_key(
            "runway",
            d.get("RUNWAY_mvt"),
            d.get("ADEP_mvt"),
        )

        add_edge(airport, runway)
        add_edge(airport, destination)
        add_edge(runway, destination)

    filtered = [
        (a, b, cnt)
        for (a, b), cnt in edge_counts.items()
        if cnt >= min_edge_count
    ]

    nodes = sorted(
        set(a for a, _, _ in filtered)
        | set(b for _, b, _ in filtered)
    )

    if len(nodes) < 3:
        raise ValueError("Static graph produced too few nodes.")

    node_to_idx = {
        n: i
        for i, n in enumerate(nodes)
    }

    rows = []
    cols_idx = []
    vals = []

    for a, b, cnt in filtered:
        ia = node_to_idx[a]
        ib = node_to_idx[b]
        weight = float(np.log1p(cnt))

        rows.extend([ia, ib])
        cols_idx.extend([ib, ia])
        vals.extend([weight, weight])

    n_nodes = len(nodes)

    A = sparse.csr_matrix(
        (vals, (rows, cols_idx)),
        shape=(n_nodes, n_nodes),
        dtype=np.float32,
    )

    degree = np.asarray(
        A.sum(axis=1)
    ).ravel()

    inv_sqrt = np.zeros_like(
        degree,
        dtype=np.float32,
    )

    positive = degree > 0
    inv_sqrt[positive] = (
        1.0 / np.sqrt(degree[positive])
    )

    D = sparse.diags(inv_sqrt)
    A_norm = D @ A @ D

    actual_dim = int(
        min(dim, max(2, n_nodes - 1))
    )

    svd = TruncatedSVD(
        n_components=actual_dim,
        random_state=random_state,
        n_iter=7,
    )

    embedding = svd.fit_transform(
        A_norm
    ).astype("float32")

    print(
        f"Static graph: {n_nodes:,} nodes, "
        f"{len(filtered):,} edges, "
        f"dim={actual_dim}"
    )

    print(
        "Static graph explained SVD variance:",
        f"{svd.explained_variance_ratio_.sum():.3f}"
    )

    return {
        "node_to_idx": node_to_idx,
        "embedding": embedding,
        "dim": actual_dim,
    }


def apply_graph_embeddings(df, state):
    """
    Add only static graph engineered columns.
    """
    out = df.copy(deep=False)

    node_to_idx = state["node_to_idx"]
    embedding = state["embedding"]
    dim = state["dim"]

    entity_specs = [
        ("airport", "ADEP_mvt", False),
        ("destination", "ADES_mvt", False),
        ("runway", "RUNWAY_mvt", True),
    ]

    airport_values = (
        out["ADEP_mvt"].to_numpy()
        if "ADEP_mvt" in out.columns
        else np.array(["__MISSING__"] * len(out), dtype=object)
    )

    generated = {}

    for kind, col, needs_airport in entity_specs:
        mat = np.zeros(
            (len(out), dim),
            dtype=np.float32,
        )

        if col in out.columns:
            vals = out[col].to_numpy()
        else:
            vals = np.array(
                ["__MISSING__"] * len(out),
                dtype=object,
            )

        for i, val in enumerate(vals):
            key = _graph_node_key(
                kind,
                val,
                airport_values[i] if needs_airport else None,
            )

            idx = node_to_idx.get(key)

            if idx is not None:
                mat[i] = embedding[idx]

        for j in range(dim):
            generated[f"g_{kind}_{j:02d}"] = mat[:, j]

    # Assign in one concat to avoid DataFrame fragmentation.
    graph_frame = pd.DataFrame(
        generated,
        index=out.index,
    )

    out = pd.concat(
        [out, graph_frame],
        axis=1,
        copy=False,
    )

    airport_cols = [
        f"g_airport_{j:02d}"
        for j in range(dim)
    ]
    runway_cols = [
        f"g_runway_{j:02d}"
        for j in range(dim)
    ]
    destination_cols = [
        f"g_destination_{j:02d}"
        for j in range(dim)
    ]

    out["g_airport_runway_dot"] = (
        out[airport_cols].to_numpy(dtype=np.float32, copy=False)
        * out[runway_cols].to_numpy(dtype=np.float32, copy=False)
    ).sum(axis=1).astype("float32")

    out["g_airport_destination_dot"] = (
        out[airport_cols].to_numpy(dtype=np.float32, copy=False)
        * out[destination_cols].to_numpy(dtype=np.float32, copy=False)
    ).sum(axis=1).astype("float32")

    out["g_runway_destination_dot"] = (
        out[runway_cols].to_numpy(dtype=np.float32, copy=False)
        * out[destination_cols].to_numpy(dtype=np.float32, copy=False)
    ).sum(axis=1).astype("float32")

    return out

In [ ]:
#@title 8. October development split + priors + static graph state

event_time = to_utc(train_dep["MVT_TIME_UTC_mvt"])

DEV_START = pd.Timestamp("2025-10-01", tz="UTC")
DEV_END = pd.Timestamp("2025-11-01", tz="UTC")

dev_train = train_dep.loc[
    event_time < DEV_START
].copy()

dev_valid = train_dep.loc[
    (event_time >= DEV_START)
    & (event_time < DEV_END)
].copy()

print(
    "Development train:",
    f"{len(dev_train):,}"
)

print(
    "October validation:",
    f"{len(dev_valid):,}"
)

# ------------------------------------------------------------
# General leakage-safe historical prior state.
# Fallback-only; prior_* columns are excluded from CatBoost.
# ------------------------------------------------------------
DEV_PRIOR_STATE = fit_target_priors(
    dev_train
)

dev_train = apply_target_priors(
    dev_train,
    DEV_PRIOR_STATE,
)

dev_valid = apply_target_priors(
    dev_valid,
    DEV_PRIOR_STATE,
)

# ------------------------------------------------------------
# Restore the V7.5 static SVD graph state.
# Fit ONLY on development training rows.
# ------------------------------------------------------------
DEV_GRAPH_STATE = build_graph_embedding_state(
    dev_train,
    dim=GRAPH_EMBED_DIM,
    min_edge_count=GRAPH_MIN_EDGE_COUNT,
    random_state=SEED,
)

dev_train = apply_graph_embeddings(
    dev_train,
    DEV_GRAPH_STATE,
)

dev_valid = apply_graph_embeddings(
    dev_valid,
    DEV_GRAPH_STATE,
)

print(
    "✓ General historical priors added "
    "(fallback only; excluded from CatBoost)."
)

print(
    "✓ Static SVD g_* graph features restored."
)

print(
    "✓ Dynamic tg_* temporal graph features retained."
)

In [ ]:
#@title 9. NM-anchor masks + missing fallback candidates

def add_unified_baseline(df, training_global_median=None):
    out = add_nm_anchor(df)

    fallback = pd.to_numeric(
        out["prior_airport_runway_hour"],
        errors="coerce",
    )
    fallback = fallback.fillna(
        pd.to_numeric(out["prior_airport_runway"], errors="coerce")
    )
    fallback = fallback.fillna(
        pd.to_numeric(out["prior_airport"], errors="coerce")
    )

    if training_global_median is None:
        training_global_median = 900.0

    fallback = fallback.fillna(float(training_global_median))

    nm = pd.to_numeric(out["nm_anchor_sec"], errors="coerce")

    out["baseline_sec"] = nm.where(nm.notna(), fallback).astype("float32")
    out["baseline_is_nm"] = nm.notna().astype("int8")
    out["baseline_is_prior"] = nm.isna().astype("int8")

    return out

DEV_GLOBAL_MEDIAN = float(dev_train[TARGET].median())
DEV_MISSING_STATE = fit_missing_fallback_state(dev_train)

print(
    "Missing-anchor training rows:",
    f"{DEV_MISSING_STATE['n_missing_train']:,}"
)

dev_train_u = add_unified_baseline(dev_train, DEV_GLOBAL_MEDIAN)
dev_valid_u = add_unified_baseline(dev_valid, DEV_GLOBAL_MEDIAN)

train_known_mask_full = dev_train_u["baseline_is_nm"].eq(1).to_numpy()
valid_known_mask_full = dev_valid_u["baseline_is_nm"].eq(1).to_numpy()
valid_missing_mask_full = ~valid_known_mask_full

print(
    "NM baseline availability — train:",
    f"{train_known_mask_full.mean()*100:.2f}%"
)
print(
    "NM baseline availability — valid:",
    f"{valid_known_mask_full.mean()*100:.2f}%"
)

y_valid_all = dev_valid_u[TARGET].astype("float32").to_numpy(copy=True)
baseline_valid_all = dev_valid_u["baseline_sec"].astype("float32").to_numpy(copy=True)

valid_missing_df = dev_valid_u.loc[
    dev_valid_u["baseline_is_nm"].eq(0)
].copy()

general_prior_missing = valid_missing_df["baseline_sec"].astype("float64").to_numpy(copy=True)

missing_hier_median = predict_missing_hierarchy(
    valid_missing_df,
    DEV_MISSING_STATE,
    statistic="median",
)

missing_hier_mean = predict_missing_hierarchy(
    valid_missing_df,
    DEV_MISSING_STATE,
    statistic="mean",
)

y_train_residual_full = (
    dev_train_u[TARGET].astype("float32")
    - dev_train_u["nm_anchor_sec"].astype("float32")
).to_numpy()

y_valid_residual_full = (
    dev_valid_u[TARGET].astype("float32")
    - dev_valid_u["nm_anchor_sec"].astype("float32")
).to_numpy()

report_ram("after NM masks and fallback candidates")

### V7.4 low-RAM / clean-learning strategy

For October development:

- all movements are used to construct traffic context
- dynamic graph features are stored only for departures
- full movement tables are released after graph extraction
- CatBoost training matrices contain only NM-anchor rows
- the ~1% missing-anchor rows do not enter CatBoost training
- missing-anchor predictions use the leakage-safe historical-prior fallback
- training matrix construction remains in-place
- static graph embedding dimension remains 8

This simultaneously reduces RAM use and restores the cleaner residual-learning problem.

In [ ]:
#@title 10. V17.2 FINAL ONE-RUN peer-imputed missing anchor + shared known-anchor matrix build

report_ram(
    "start V17 matrix workflow"
)

# ============================================================
# PART A — PEER-IMPUTED MISSING-ANCHOR RECOVERY
# ============================================================

FROZEN_V9_MISSING_RMSE = 888.5981276385777
FROZEN_V9_MISSING_MAE = 381.294652
FROZEN_V9_MISSING_CONFIG = {
    "w_hist": 0.1,
    "w_global": 0.9,
    "w_missing_only": 0.0,
    "clip_max": None,
}

ACTIVE_MISSING_RMSE = FROZEN_V9_MISSING_RMSE
ACTIVE_MISSING_MAE = FROZEN_V9_MISSING_MAE
ACTIVE_MISSING_PRED = None
ACTIVE_MISSING_NAME = "frozen_v9"

def _peer_calendar_frame(df):
    """Minimal calendar/key frame for peer statistics."""
    out = pd.DataFrame(
        index=df.index
    )

    mvt = to_utc(
        df["MVT_TIME_UTC_mvt"]
    )

    sched = to_utc(
        df["SCHED_TIME_UTC_mvt"]
    )

    # Prefer actual movement calendar, fall back to scheduled time.
    t = mvt.fillna(
        sched
    )

    out["_airport"] = (
        df["ADEP_mvt"]
        .astype("string")
        .fillna("__MISSING__")
        .astype(str)
    )

    out["_runway"] = (
        df["RUNWAY_mvt"]
        .astype("string")
        .fillna("__MISSING__")
        .astype(str)
    )

    out["_date"] = (
        t.dt.strftime("%Y-%m-%d")
        .fillna("__MISSING__")
    )

    out["_month"] = (
        t.dt.strftime("%Y-%m")
        .fillna("__MISSING__")
    )

    out["_week"] = (
        t.dt.strftime("%G-W%V")
        .fillna("__MISSING__")
    )

    out["_hour"] = (
        t.dt.hour
        .fillna(-1)
        .astype("int16")
    )

    out["_dow"] = (
        t.dt.dayofweek
        .fillna(-1)
        .astype("int8")
    )

    return out


PEER_LEVELS = [
    (
        "airport_runway_date_hour",
        ["_airport", "_runway", "_date", "_hour"],
        6,
    ),
    (
        "airport_runway_date",
        ["_airport", "_runway", "_date"],
        12,
    ),
    (
        "airport_date_hour",
        ["_airport", "_date", "_hour"],
        12,
    ),
    (
        "airport_date",
        ["_airport", "_date"],
        25,
    ),
    (
        "airport_runway_week_hour",
        ["_airport", "_runway", "_week", "_hour"],
        18,
    ),
    (
        "airport_runway_month_hour",
        ["_airport", "_runway", "_month", "_hour"],
        30,
    ),
    (
        "airport_runway_month",
        ["_airport", "_runway", "_month"],
        60,
    ),
    (
        "airport_month_hour",
        ["_airport", "_month", "_hour"],
        60,
    ),
    (
        "airport_month_dow",
        ["_airport", "_month", "_dow"],
        80,
    ),
    (
        "airport_month",
        ["_airport", "_month"],
        150,
    ),
    (
        "airport",
        ["_airport"],
        300,
    ),
]


def build_peer_anchor_features(
    full_df,
    target_mask,
    audit=False,
):
    """
    Build peer features for selected rows using ONLY known-anchor peers
    from the same dataframe/split.

    This is transductive but label-free: peer values come from observable
    NM anchor timestamps and MVT_TIME, never TAXITIME targets.
    """
    key = _peer_calendar_frame(
        full_df
    )

    known_mask = (
        full_df["baseline_is_nm"]
        .eq(1)
        .to_numpy()
    )

    peer_anchor = pd.to_numeric(
        full_df["nm_anchor_sec"],
        errors="coerce",
    )

    source = key.loc[
        known_mask
    ].copy()

    source["_anchor"] = (
        peer_anchor.loc[
            source.index
        ]
        .astype("float64")
    )

    source = source.loc[
        source["_anchor"].notna()
    ].copy()

    # Robust winsorization based only on observable peer-anchor values.
    lo = float(
        source["_anchor"].quantile(
            0.005
        )
    )

    hi = float(
        source["_anchor"].quantile(
            0.995
        )
    )

    if not np.isfinite(lo):
        lo = 0.0

    if not np.isfinite(hi):
        hi = 7200.0

    if hi <= lo:
        lo, hi = 0.0, 7200.0

    source["_anchor_robust"] = (
        source["_anchor"]
        .clip(
            lower=lo,
            upper=hi,
        )
    )

    target = key.loc[
        target_mask
    ].copy()

    result = pd.DataFrame(
        index=target.index
    )

    # Calendar features themselves help the correction model.
    result["peer_hour"] = (
        target["_hour"]
        .astype("float32")
    )

    result["peer_dow"] = (
        target["_dow"]
        .astype("float32")
    )

    result["peer_month_num"] = (
        pd.to_datetime(
            target["_month"] + "-01",
            errors="coerce",
        )
        .dt.month
        .fillna(-1)
        .astype("float32")
    )

    selected = pd.Series(
        np.nan,
        index=target.index,
        dtype="float64",
    )

    selected_level = pd.Series(
        "__NONE__",
        index=target.index,
        dtype="object",
    )

    selected_count = pd.Series(
        0.0,
        index=target.index,
        dtype="float64",
    )

    for level_name, keys, min_count in PEER_LEVELS:
        stats = (
            source
            .groupby(
                keys,
                sort=False,
                observed=True,
            )["_anchor_robust"]
            .agg(
                ["median", "mean", "count"]
            )
            .reset_index()
        )

        stats = stats.rename(
            columns={
                "median": f"peer_{level_name}_median",
                "mean": f"peer_{level_name}_mean",
                "count": f"peer_{level_name}_count",
            }
        )

        merged = (
            target[
                keys
            ]
            .reset_index()
            .merge(
                stats,
                how="left",
                on=keys,
                sort=False,
            )
            .set_index(
                "index"
            )
            .reindex(
                target.index
            )
        )

        med_col = (
            f"peer_{level_name}_median"
        )

        mean_col = (
            f"peer_{level_name}_mean"
        )

        count_col = (
            f"peer_{level_name}_count"
        )

        result[med_col] = pd.to_numeric(
            merged[med_col],
            errors="coerce",
        ).astype("float32")

        result[mean_col] = pd.to_numeric(
            merged[mean_col],
            errors="coerce",
        ).astype("float32")

        result[count_col] = (
            pd.to_numeric(
                merged[count_col],
                errors="coerce",
            )
            .fillna(0)
            .astype("float32")
        )

        usable = (
            selected.isna()
            & result[med_col].notna()
            & (
                result[count_col]
                >= float(
                    min_count
                )
            )
        )

        selected.loc[
            usable
        ] = result.loc[
            usable,
            med_col,
        ].astype("float64")

        selected_level.loc[
            usable
        ] = level_name

        selected_count.loc[
            usable
        ] = result.loc[
            usable,
            count_col,
        ].astype("float64")

        del stats, merged
        gc.collect()

    # Final global observable-anchor fallback if no hierarchy level qualifies.
    global_peer_median = float(
        source["_anchor_robust"].median()
    )

    selected = selected.fillna(
        global_peer_median
    )

    selected_level = selected_level.where(
        selected_level.ne(
            "__NONE__"
        ),
        "__GLOBAL__",
    )

    result[
        "peer_pseudo_anchor_sec"
    ] = selected.astype(
        "float32"
    )

    result[
        "peer_selected_count"
    ] = selected_count.astype(
        "float32"
    )

    result[
        "peer_selected_level"
    ] = selected_level.astype(
        str
    )

    # Dispersion proxies from finest useful airport/runway/month level.
    for candidate in [
        "airport_runway_month_hour",
        "airport_runway_month",
        "airport_month_hour",
        "airport_month",
        "airport",
    ]:
        med_col = (
            f"peer_{candidate}_median"
        )

        mean_col = (
            f"peer_{candidate}_mean"
        )

        if (
            med_col in result.columns
            and mean_col in result.columns
        ):
            result[
                f"peer_{candidate}_mean_minus_median"
            ] = (
                result[mean_col]
                - result[med_col]
            ).astype(
                "float32"
            )

    if audit:
        print(
            "Peer anchor winsorization:",
            {
                "lo": lo,
                "hi": hi,
            }
        )

        print(
            "Peer source rows:",
            f"{len(source):,}"
        )

        print(
            "Peer target rows:",
            f"{len(target):,}"
        )

        print(
            "Selected peer levels:"
        )

        display(
            result[
                "peer_selected_level"
            ]
            .value_counts(
                dropna=False
            )
            .rename_axis(
                "level"
            )
            .reset_index(
                name="rows"
            )
        )

    del source, target, key
    gc.collect()

    return result



PEER_DROP_EXACT = {
    "nm_anchor_sec",
    "nm_anchor_source",
    "nm_anchor_missing",
    "aobt_minus_sched_min",
    "eobt_minus_sched_min",
    "iobt_minus_sched_min",
    "lobt_minus_sched_min",
    "lobt_minus_iobt_min",
    "eobt_minus_iobt_min",
}

PEER_DROP_PREFIXES = (
    "aobt_",
    "eobt_",
    "iobt_",
    "lobt_",
    "nm_anchor",
)


def select_pseudo_missing_indices(
    full_df,
    n_rows,
    random_state,
):
    """
    Sample known-anchor rows to masquerade as missing rows.

    Sampling is biased toward airport/month combinations where genuine
    missing anchors occur, so the synthetic regime resembles the real one.
    """
    known_mask = (
        full_df[
            "baseline_is_nm"
        ]
        .eq(
            1
        )
    )

    missing_mask = (
        ~known_mask
    )

    known_idx = full_df.index[
        known_mask
    ]

    if len(
        known_idx
    ) == 0:
        return pd.Index(
            []
        )

    n_rows = int(
        min(
            n_rows,
            len(
                known_idx
            ),
        )
    )

    mvt = to_utc(
        full_df[
            "MVT_TIME_UTC_mvt"
        ]
    )

    airport = (
        full_df[
            "ADEP_mvt"
        ]
        .astype(
            "string"
        )
        .fillna(
            "__MISSING__"
        )
    )

    month = (
        mvt.dt.strftime(
            "%Y-%m"
        )
        .fillna(
            "__MISSING__"
        )
    )

    key = (
        airport
        + "|"
        + month
    )

    missing_counts = (
        key.loc[
            missing_mask
        ]
        .value_counts()
    )

    known_counts = (
        key.loc[
            known_mask
        ]
        .value_counts()
    )

    if len(
        missing_counts
    ) == 0:
        weights = None

    else:
        # Desired density / available density.
        ratio = (
            (
                missing_counts
                / max(
                    1,
                    missing_counts.sum(),
                )
            )
            / (
                known_counts
                / max(
                    1,
                    known_counts.sum(),
                )
            )
        ).replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )

        known_key = key.loc[
            known_idx
        ]

        weights = (
            known_key
            .map(
                ratio
            )
            .fillna(
                0.05
            )
            .clip(
                lower=0.01,
                upper=50.0,
            )
        )

    sample_frame = pd.DataFrame(
        index=known_idx
    )

    sampled = (
        sample_frame
        .sample(
            n=n_rows,
            replace=False,
            weights=weights,
            random_state=random_state,
        )
        .index
    )

    return sampled


def prepare_augmented_missing_training(
    full_df,
    pseudo_rows,
    random_state,
    audit=False,
):
    """
    Real missing rows + synthetic missing rows.

    For synthetic rows, all NM off-block timestamps are hidden before feature
    construction. The synthetic row is also removed from the peer-source pool,
    preventing its own known anchor from leaking into its peer features.
    """
    real_missing_mask = (
        full_df[
            "baseline_is_nm"
        ]
        .eq(
            0
        )
    )

    real_idx = full_df.index[
        real_missing_mask
    ]

    synth_idx = select_pseudo_missing_indices(
        full_df,
        n_rows=pseudo_rows,
        random_state=random_state,
    )

    # Minimal peer-source frame.
    peer_cols = [
        c
        for c in [
            "MVT_TIME_UTC_mvt",
            "SCHED_TIME_UTC_mvt",
            "ADEP_mvt",
            "RUNWAY_mvt",
            "baseline_is_nm",
            "nm_anchor_sec",
        ]
        if c in full_df.columns
    ]

    source_view = (
        full_df[
            peer_cols
        ]
        .copy()
    )

    # Hide each synthetic target from the observable-anchor peer source.
    if len(
        synth_idx
    ):
        source_view.loc[
            synth_idx,
            "baseline_is_nm",
        ] = 0

        source_view.loc[
            synth_idx,
            "nm_anchor_sec",
        ] = np.nan

    target_mask = (
        source_view.index.isin(
            real_idx
        )
        | source_view.index.isin(
            synth_idx
        )
    )

    peer_features = build_peer_anchor_features(
        source_view,
        target_mask,
        audit=audit,
    )

    real_df = (
        full_df.loc[
            real_idx
        ]
        .copy()
    )

    synth_df = (
        full_df.loc[
            synth_idx
        ]
        .copy()
    )

    # Make synthetic rows look genuinely missing to every downstream feature.
    for col in [
        "AOBT_3_flt",
        "EOBT_1_flt",
        "IOBT_flt",
        "LOBT_flt",
    ]:
        if col in synth_df.columns:
            synth_df[
                col
            ] = pd.NaT

    for col in [
        "nm_anchor_sec",
    ]:
        if col in synth_df.columns:
            synth_df[
                col
            ] = np.nan

    if (
        "nm_anchor_source"
        in synth_df.columns
    ):
        synth_df[
            "nm_anchor_source"
        ] = "__MISSING__"

    if (
        "nm_anchor_missing"
        in synth_df.columns
    ):
        synth_df[
            "nm_anchor_missing"
        ] = 1

    if (
        "baseline_is_nm"
        in synth_df.columns
    ):
        synth_df[
            "baseline_is_nm"
        ] = 0

    if (
        "baseline_is_prior"
        in synth_df.columns
    ):
        synth_df[
            "baseline_is_prior"
        ] = 1

    train_aug = pd.concat(
        [
            real_df,
            synth_df,
        ],
        axis=0,
        copy=False,
    )

    for c in peer_features.columns:
        train_aug[
            c
        ] = (
            peer_features[
                c
            ]
            .reindex(
                train_aug.index
            )
        )

    sample_weight = np.where(
        train_aug.index.isin(
            real_idx
        ),
        float(
            PSEUDO_MISSING_REAL_WEIGHT
        ),
        float(
            PSEUDO_MISSING_SYNTH_WEIGHT
        ),
    ).astype(
        np.float32
    )

    if audit:
        print(
            "Real missing training rows:",
            f"{len(real_idx):,}"
        )

        print(
            "Synthetic missing rows:",
            f"{len(synth_idx):,}"
        )

        print(
            "Augmented missing rows:",
            f"{len(train_aug):,}"
        )

    del (
        source_view,
        real_df,
        synth_df,
        peer_features,
    )

    gc.collect()

    return (
        train_aug,
        sample_weight,
        len(
            real_idx
        ),
        len(
            synth_idx
        ),
    )


def attach_peer_features_for_targets(
    full_df,
    target_mask,
    audit=False,
):
    target_df = (
        full_df.loc[
            target_mask
        ]
        .copy()
    )

    peer = build_peer_anchor_features(
        full_df,
        target_mask,
        audit=audit,
    )

    for c in peer.columns:
        target_df[
            c
        ] = (
            peer[
                c
            ]
            .reindex(
                target_df.index
            )
        )

    del peer
    gc.collect()

    return target_df


def add_missing_prior_features_inplace(
    train_df,
    other_df=None,
):
    for col in [
        "prior_airport_runway_hour",
        "prior_airport_runway",
        "prior_airport",
    ]:
        if col in train_df.columns:
            train_df[
                "mp_" + col
            ] = pd.to_numeric(
                train_df[
                    col
                ],
                errors="coerce",
            ).astype(
                "float32"
            )

        if (
            other_df is not None
            and col in other_df.columns
        ):
            other_df[
                "mp_" + col
            ] = pd.to_numeric(
                other_df[
                    col
                ],
                errors="coerce",
            ).astype(
                "float32"
            )

    return train_df


if RUN_PEER_MISSING_BRANCH:
    print()
    print(
        "=== V17 AUGMENTED PEER-IMPUTED MISSING BRANCH ==="
    )

    train_missing_mask_full = (
        ~train_known_mask_full
    )

    # Genuine + synthetic missing-anchor training data.
    (
        missing_train_df,
        peer_sample_weight,
        n_real_missing_train,
        n_synth_missing_train,
    ) = prepare_augmented_missing_training(
        dev_train_u,
        pseudo_rows=PSEUDO_MISSING_ROWS,
        random_state=SEED + 1700,
        audit=RUN_PEER_MISSING_AUDIT,
    )

    # Validation stays genuine missing only.
    missing_valid_df = (
        attach_peer_features_for_targets(
            dev_valid_u,
            valid_missing_mask_full,
            audit=RUN_PEER_MISSING_AUDIT,
        )
    )

    y_peer_train = (
        missing_train_df[
            TARGET
        ]
        .astype(
            "float32"
        )
        .to_numpy(
            copy=True
        )
    )

    y_peer_valid = (
        missing_valid_df[
            TARGET
        ]
        .astype(
            "float64"
        )
        .to_numpy(
            copy=True
        )
    )

    add_missing_prior_features_inplace(
        missing_train_df,
        missing_valid_df,
    )

    pseudo_train = (
        pd.to_numeric(
            missing_train_df[
                "peer_pseudo_anchor_sec"
            ],
            errors="coerce",
        )
        .fillna(
            DEV_GLOBAL_MEDIAN
        )
        .to_numpy(
            dtype=np.float32,
            copy=True,
        )
    )

    pseudo_valid = (
        pd.to_numeric(
            missing_valid_df[
                "peer_pseudo_anchor_sec"
            ],
            errors="coerce",
        )
        .fillna(
            DEV_GLOBAL_MEDIAN
        )
        .to_numpy(
            dtype=np.float64,
            copy=True,
        )
    )

    X_peer_train, PEER_FEATURES, PEER_CATS = (
        make_original_method_b_plus_graph_inplace(
            missing_train_df
        )
    )

    PEER_FEATURES = list(
        X_peer_train.columns
    )

    # Missing branch must not consume direct NM timing.
    PEER_DROP_EXACT = {
        "nm_anchor_sec",
        "nm_anchor_source",
        "nm_anchor_missing",
        "aobt_minus_sched_min",
        "eobt_minus_sched_min",
        "iobt_minus_sched_min",
        "lobt_minus_sched_min",
        "lobt_minus_iobt_min",
        "eobt_minus_iobt_min",
    }

    PEER_DROP_PREFIXES = (
        "aobt_",
        "eobt_",
        "iobt_",
        "lobt_",
        "nm_anchor",
    )

    peer_drop = [
        c
        for c in PEER_FEATURES
        if (
            c in PEER_DROP_EXACT
            or (
                c.startswith(
                    PEER_DROP_PREFIXES
                )
                and not c.startswith(
                    "mp_"
                )
            )
        )
    ]

    X_peer_train.drop(
        columns=peer_drop,
        inplace=True,
        errors="ignore",
    )

    PEER_FEATURES = list(
        X_peer_train.columns
    )

    PEER_CATS = [
        c
        for c in PEER_CATS
        if c in PEER_FEATURES
    ]

    if (
        "peer_selected_level"
        in PEER_FEATURES
        and "peer_selected_level"
        not in PEER_CATS
    ):
        PEER_CATS.append(
            "peer_selected_level"
        )

    X_peer_valid, _, _ = (
        make_original_method_b_plus_graph_inplace(
            missing_valid_df
        )
    )

    X_peer_valid.drop(
        columns=[
            c
            for c in X_peer_valid.columns
            if c not in PEER_FEATURES
        ],
        inplace=True,
        errors="ignore",
    )

    missing_cols = [
        c
        for c in PEER_FEATURES
        if c not in X_peer_valid.columns
    ]

    if missing_cols:
        raise ValueError(
            f"Peer validation missing features: {missing_cols[:20]}"
        )

    X_peer_valid = (
        X_peer_valid.reindex(
            columns=PEER_FEATURES
        )
    )

    if RUN_PEER_MISSING_AUDIT:
        print(
            "Peer model features:",
            len(
                PEER_FEATURES
            )
        )

        print(
            "Peer categoricals:",
            len(
                PEER_CATS
            )
        )

        audit_feature_types(
            X_peer_train,
            PEER_CATS,
            "V17 augmented peer-missing matrix",
        )

    y_peer_correction_train = (
        y_peer_train
        - pseudo_train
    ).astype(
        np.float32
    )

    y_peer_correction_valid = (
        y_peer_valid
        - pseudo_valid
    ).astype(
        np.float64
    )

    for c in PEER_CATS:
        tr = (
            X_peer_train[
                c
            ]
            .astype(
                "string"
            )
            .fillna(
                "__MISSING__"
            )
        )

        categories = pd.Index(
            pd.unique(
                tr
            )
        )

        X_peer_train[
            c
        ] = pd.Categorical(
            tr,
            categories=categories,
        )

        va = (
            X_peer_valid[
                c
            ]
            .astype(
                "string"
            )
            .fillna(
                "__MISSING__"
            )
        )

        X_peer_valid[
            c
        ] = pd.Categorical(
            va,
            categories=categories,
        )

    peer_model = LGBMRegressor(
        objective="regression",
        metric="rmse",
        n_estimators=1200,
        learning_rate=0.020,
        num_leaves=48,
        max_depth=-1,
        min_child_samples=120,
        subsample=0.90,
        subsample_freq=1,
        colsample_bytree=0.82,
        reg_alpha=0.10,
        reg_lambda=24.0,
        max_bin=255,
        device_type="cpu",
        random_state=SEED + 1701,
        n_jobs=-1,
        verbosity=-1,
    )

    print()
    print(
        "Training V17 augmented peer-correction LightGBM..."
    )

    peer_model.fit(
        X_peer_train,
        y_peer_correction_train,
        sample_weight=peer_sample_weight,
        categorical_feature=PEER_CATS,
        eval_set=[
            (
                X_peer_valid,
                y_peer_correction_valid,
            )
        ],
        callbacks=[
            lgb_early_stopping(
                100,
                verbose=True,
            ),
            lgb_log_evaluation(
                100
            ),
        ],
    )

    PEER_FIXED_ITERS = int(
        min(
            1200,
            getattr(
            peer_model,
            "best_iteration_",
            0,
        )
            or 900
        )
    )

    pred_peer_corr = (
        peer_model.predict(
            X_peer_valid,
            num_iteration=PEER_FIXED_ITERS,
        )
        .astype(
            np.float64
        )
    )

    pred_peer_corrected = (
        pseudo_valid
        + pred_peer_corr
    )

    pred_peer_raw = (
        pseudo_valid.copy()
    )

    pred_hist = (
        0.50
        * general_prior_missing
        + 0.50
        * missing_hier_mean
    ).astype(
        np.float64
    )

    print()
    print(
        "=== V17 RAW MISSING CANDIDATES ==="
    )

    raw_peer_rows = []

    for name, p in {
        "peer_pseudo_anchor": pred_peer_raw,
        "peer_corrected_augmented_lgbm": pred_peer_corrected,
        "historical_50_50": pred_hist,
    }.items():
        raw_peer_rows.append({
            "candidate": name,
            "rmse_sec": (
                mean_squared_error(
                    y_peer_valid,
                    p,
                )
                ** 0.5
            ),
            "mae_sec": mean_absolute_error(
                y_peer_valid,
                p,
            ),
        })

    raw_peer_scores = (
        pd.DataFrame(
            raw_peer_rows
        )
        .sort_values(
            "rmse_sec"
        )
    )

    display(
        raw_peer_scores
    )

    blend_rows = []
    best_peer_rmse = float(
        "inf"
    )
    best_peer_pred = None
    BEST_PEER_CONFIG = None

    weights = np.arange(
        0.0,
        1.0001,
        0.05,
    )

    for w_corr in weights:
        for w_peer in weights:
            w_hist = (
                1.0
                - w_corr
                - w_peer
            )

            if w_hist < -1e-9:
                continue

            w_hist = max(
                0.0,
                float(
                    w_hist
                ),
            )

            base_pred = (
                float(
                    w_corr
                )
                * pred_peer_corrected
                + float(
                    w_peer
                )
                * pred_peer_raw
                + w_hist
                * pred_hist
            )

            for clip_max in [
                None,
                3600.0,
                4500.0,
                7200.0,
            ]:
                if clip_max is None:
                    p = base_pred
                else:
                    p = np.clip(
                        base_pred,
                        0.0,
                        clip_max,
                    )

                rmse = (
                    mean_squared_error(
                        y_peer_valid,
                        p,
                    )
                    ** 0.5
                )

                mae = mean_absolute_error(
                    y_peer_valid,
                    p,
                )

                row = {
                    "w_corrected": float(
                        w_corr
                    ),
                    "w_peer_raw": float(
                        w_peer
                    ),
                    "w_hist": float(
                        w_hist
                    ),
                    "clip_max": clip_max,
                    "rmse_sec": float(
                        rmse
                    ),
                    "mae_sec": float(
                        mae
                    ),
                }

                blend_rows.append(
                    row
                )

                if rmse < best_peer_rmse:
                    best_peer_rmse = float(
                        rmse
                    )

                    best_peer_pred = (
                        p.copy()
                    )

                    BEST_PEER_CONFIG = dict(
                        row
                    )

    peer_blend_scores = (
        pd.DataFrame(
            blend_rows
        )
        .sort_values(
            "rmse_sec"
        )
    )

    print()
    print(
        "=== BEST V17 MISSING BLENDS ==="
    )

    display(
        peer_blend_scores.head(
            20
        )
    )

    PEER_MISSING_RMSE = float(
        best_peer_rmse
    )

    PEER_MISSING_PRED = (
        best_peer_pred
    )

    PEER_MISSING_MAE = (
        mean_absolute_error(
            y_peer_valid,
            PEER_MISSING_PRED,
        )
    )

    print(
        "Best peer config:",
        BEST_PEER_CONFIG,
    )

    print(
        "Peer model fixed iterations:",
        PEER_FIXED_ITERS,
    )

    print(
        f"Peer missing RMSE: {PEER_MISSING_RMSE:.3f} sec"
    )

    print(
        f"Gain vs frozen V9: "
        f"{FROZEN_V9_MISSING_RMSE - PEER_MISSING_RMSE:+.3f} sec"
    )

    if (
        PEER_MISSING_RMSE
        < FROZEN_V9_MISSING_RMSE
    ):
        ACTIVE_MISSING_RMSE = (
            PEER_MISSING_RMSE
        )

        ACTIVE_MISSING_MAE = (
            PEER_MISSING_MAE
        )

        ACTIVE_MISSING_PRED = (
            PEER_MISSING_PRED
        )

        ACTIVE_MISSING_NAME = (
            "v17_augmented_peer"
        )

    else:
        print(
            "V17 augmented peer branch did not beat the frozen V9 benchmark "
            "on October. Validation overall keeps frozen V9 metrics, but "
            "production can still generate row-level missing predictions."
        )

    del (
        X_peer_train,
        X_peer_valid,
        peer_model,
        missing_train_df,
        missing_valid_df,
        peer_sample_weight,
    )

    gc.collect()

    report_ram(
        "after V17 augmented missing branch"
    )

else:
    PEER_FIXED_ITERS = 900
    BEST_PEER_CONFIG = {
        "w_corrected": 0.90,
        "w_peer_raw": 0.00,
        "w_hist": 0.10,
        "clip_max": None,
    }

    print(
        "RUN_PEER_MISSING_BRANCH=False -> using frozen V9 missing benchmark."
    )

# ============================================================
# PART B — REBUILD KNOWN-ANCHOR ENSEMBLE BRANCH
# ============================================================

# dev_train / dev_valid were not modified by the direct in-place selector.
# Recreate unified frames cleanly from them.
dev_train_u = add_unified_baseline(
    dev_train,
    DEV_GLOBAL_MEDIAN,
)

dev_valid_u = add_unified_baseline(
    dev_valid,
    DEV_GLOBAL_MEDIAN,
)

# Compact residual targets from clean rebuilt frames.
train_known_mask_rebuilt = (
    dev_train_u["baseline_is_nm"]
    .eq(1)
    .to_numpy()
)

valid_known_mask_rebuilt = (
    dev_valid_u["baseline_is_nm"]
    .eq(1)
    .to_numpy()
)

y_train_residual = (
    (
        dev_train_u[TARGET].astype("float32")
        - dev_train_u["nm_anchor_sec"].astype("float32")
    )
    .to_numpy()[
        train_known_mask_rebuilt
    ]
    .astype("float32", copy=False)
)

y_valid_residual = (
    (
        dev_valid_u[TARGET].astype("float32")
        - dev_valid_u["nm_anchor_sec"].astype("float32")
    )
    .to_numpy()[
        valid_known_mask_rebuilt
    ]
    .astype("float32", copy=False)
)

# Verify masks match the preserved master masks.
assert np.array_equal(
    valid_known_mask_rebuilt,
    valid_known_mask_full,
)

# We no longer need duplicate base frames after rebuild.
for _name in [
    "dev_train",
    "dev_valid",
]:
    if _name in globals():
        del globals()[_name]

for _name in [
    "train_dep",
    "ranking_dep",
]:
    if _name in globals():
        del globals()[_name]

gc.collect()

# Keep only known-anchor rows.
dev_train_u.drop(
    index=dev_train_u.index[
        ~dev_train_u["baseline_is_nm"].eq(1)
    ],
    inplace=True,
)

dev_valid_u.drop(
    index=dev_valid_u.index[
        ~dev_valid_u["baseline_is_nm"].eq(1)
    ],
    inplace=True,
)

# Strong V7.7 feature matrix.
X_train, FEATURES, CATS = (
    make_original_method_b_plus_graph_inplace(
        dev_train_u
    )
)

FEATURES = list(
    X_train.columns
)

CATS = [
    c for c in CATS
    if c in FEATURES
]

del dev_train_u
gc.collect()

X_valid, _, _ = (
    make_original_method_b_plus_graph_inplace(
        dev_valid_u
    )
)

extra = [
    c for c in X_valid.columns
    if c not in FEATURES
]

if extra:
    X_valid.drop(
        columns=extra,
        inplace=True,
    )

missing = [
    c for c in FEATURES
    if c not in X_valid.columns
]

if missing:
    raise ValueError(
        f"Known validation missing features: {missing[:20]}"
    )

if list(X_valid.columns) != FEATURES:
    X_valid = X_valid.reindex(
        columns=FEATURES
    )

del dev_valid_u
gc.collect()

static_cols = [
    c for c in FEATURES
    if c.startswith("g_")
]

dynamic_cols = [
    c for c in FEATURES
    if c.startswith("tg_")
]

assert static_cols
assert dynamic_cols

audit_feature_types(
    X_train,
    CATS,
    "V17 known-anchor ensemble matrix",
)

report_ram(
    "known-anchor matrix ready"
)

In [ ]:
#@title 11. V17.2 FINAL ONE-RUN — single LightGBM, max 1200 trees

# ============================================================
# A. Shared categorical encoding
# ============================================================

print()
print(
    "Converting categoricals for LightGBM..."
)

for c in CATS:
    if c not in X_train.columns:
        continue

    tr = (
        X_train[c]
        .astype("string")
        .fillna("__MISSING__")
    )

    categories = pd.Index(
        pd.unique(
            tr
        )
    )

    X_train[c] = pd.Categorical(
        tr,
        categories=categories,
    )

    va = (
        X_valid[c]
        .astype("string")
        .fillna("__MISSING__")
    )

    X_valid[c] = pd.Categorical(
        va,
        categories=categories,
    )

    del tr, va, categories

gc.collect()

for c in X_train.columns:
    if c in CATS:
        continue

    X_train[c] = pd.to_numeric(
        X_train[c],
        errors="coerce",
        downcast="float",
    )

    X_valid[c] = pd.to_numeric(
        X_valid[c],
        errors="coerce",
        downcast="float",
    )

report_ram(
    "after V17.2 categorical conversion"
)

# ============================================================
# B. ONE LightGBM only
# ============================================================

LGB_MAX_ITERS = 1200

def make_v17_single_lgb_model(
    n_estimators=LGB_MAX_ITERS,
    seed=SEED + 3100,
):
    return LGBMRegressor(
        objective="regression",
        metric="rmse",
        n_estimators=int(
            n_estimators
        ),
        learning_rate=0.025,
        num_leaves=96,
        max_depth=-1,
        min_child_samples=120,
        min_split_gain=0.0,
        subsample=0.90,
        subsample_freq=1,
        colsample_bytree=0.88,
        reg_alpha=0.05,
        reg_lambda=12.0,
        max_bin=255,
        device_type="cpu",
        random_state=int(
            seed
        ),
        bagging_seed=int(
            seed + 101
        ),
        feature_fraction_seed=int(
            seed + 211
        ),
        data_random_seed=int(
            seed + 307
        ),
        extra_seed=int(
            seed + 401
        ),
        n_jobs=-1,
        verbosity=-1,
    )


print()
print(
    "=== TRAIN SINGLE V17.2 LIGHTGBM ==="
)

print(
    "Maximum trees:",
    LGB_MAX_ITERS
)

report_ram(
    "before single LightGBM"
)

lgb_model = make_v17_single_lgb_model()

lgb_model.fit(
    X_train,
    y_train_residual,
    categorical_feature=CATS,
    eval_set=[
        (
            X_valid,
            y_valid_residual,
        )
    ],
    callbacks=[
        lgb_early_stopping(
            100,
            verbose=True,
        ),
        lgb_log_evaluation(
            100
        ),
    ],
)

LGB_BEST_ITERS = int(
    getattr(
        lgb_model,
        "best_iteration_",
        0,
    )
    or LGB_MAX_ITERS
)

LGB_BEST_ITERS = int(
    min(
        LGB_MAX_ITERS,
        max(
            1,
            LGB_BEST_ITERS,
        ),
    )
)

pred_lgb = (
    lgb_model.predict(
        X_valid,
        num_iteration=LGB_BEST_ITERS,
    )
    .astype(
        np.float64
    )
)

lgb_rmse = (
    mean_squared_error(
        y_valid_residual,
        pred_lgb,
    )
    ** 0.5
)

lgb_mae = mean_absolute_error(
    y_valid_residual,
    pred_lgb,
)

LGB_GAIN_IMPORTANCE = (
    pd.Series(
        lgb_model.booster_
        .feature_importance(
            importance_type="gain"
        )
        .astype(
            np.float64
        ),
        index=X_train.columns,
        dtype="float64",
    )
    .sort_values(
        ascending=False
    )
)

print()
print(
    f"Single LightGBM residual RMSE: {lgb_rmse:.3f} sec"
)

print(
    f"Single LightGBM residual MAE : {lgb_mae:.3f} sec"
)

print(
    "Best iteration:",
    LGB_BEST_ITERS,
)

print()
print(
    "Top V17.2 gain features:"
)

display(
    LGB_GAIN_IMPORTANCE
    .head(
        60
    )
    .rename(
        "gain"
    )
    .rename_axis(
        "feature"
    )
    .reset_index()
)

pred_known_residual = (
    pred_lgb.copy()
)

ensemble_rmse = float(
    lgb_rmse
)

ensemble_mae = float(
    lgb_mae
)

# ============================================================
# C. Residual alignment + complete October metric
# ============================================================

known_actual = (
    y_valid_all[
        valid_known_mask_full
    ]
    .astype(
        np.float64
    )
)

known_anchor = (
    baseline_valid_all[
        valid_known_mask_full
    ]
    .astype(
        np.float64
    )
)

known_reconstructed = (
    known_anchor
    + pred_known_residual
)

residual_rmse_direct = (
    mean_squared_error(
        y_valid_residual,
        pred_known_residual,
    )
    ** 0.5
)

reconstructed_rmse = (
    mean_squared_error(
        known_actual,
        known_reconstructed,
    )
    ** 0.5
)

alignment_gap = abs(
    residual_rmse_direct
    - reconstructed_rmse
)

print()
print(
    "=== V17.2 RESIDUAL ALIGNMENT CHECK ==="
)

print(
    f"Residual-space RMSE      : {residual_rmse_direct:.9f}"
)

print(
    f"Reconstructed taxi RMSE : {reconstructed_rmse:.9f}"
)

print(
    f"Absolute difference      : {alignment_gap:.12f}"
)

if alignment_gap > 1e-4:
    raise RuntimeError(
        "Residual/reconstruction mismatch."
    )

missing_share = float(
    valid_missing_mask_full.mean()
)

known_share = (
    1.0
    - missing_share
)

known_pred_clipped = np.clip(
    known_reconstructed,
    0.0,
    None,
)

known_rmse = (
    mean_squared_error(
        known_actual,
        known_pred_clipped,
    )
    ** 0.5
)

known_mae = mean_absolute_error(
    known_actual,
    known_pred_clipped,
)

if ACTIVE_MISSING_PRED is not None:
    pred_valid_all = (
        baseline_valid_all.astype(
            np.float64,
            copy=True,
        )
    )

    pred_valid_all[
        valid_known_mask_full
    ] = known_pred_clipped

    pred_valid_all[
        valid_missing_mask_full
    ] = ACTIVE_MISSING_PRED

    overall_rmse = (
        mean_squared_error(
            y_valid_all,
            pred_valid_all,
        )
        ** 0.5
    )

    overall_mae = mean_absolute_error(
        y_valid_all,
        pred_valid_all,
    )

    overall_metric_mode = (
        "exact row-level"
    )

else:
    overall_rmse = math.sqrt(
        known_share
        * known_rmse**2
        + missing_share
        * ACTIVE_MISSING_RMSE**2
    )

    overall_mae = (
        known_share
        * known_mae
        + missing_share
        * ACTIVE_MISSING_MAE
    )

    overall_metric_mode = (
        f"{ACTIVE_MISSING_NAME} missing metrics"
    )

print()
print(
    "=" * 52
)

print(
    "V17.2 FINAL ONE-RUN RESULTS"
)

print(
    "=" * 52
)

print(
    f"Residual RMSE       : {lgb_rmse:.3f} sec"
)

print(
    f"Known-anchor RMSE   : {known_rmse:.3f} sec"
)

print(
    f"Known-anchor MAE    : {known_mae:.3f} sec"
)

print(
    f"Missing RMSE        : {ACTIVE_MISSING_RMSE:.3f} sec"
)

print(
    f"Overall RMSE        : {overall_rmse:.3f} sec"
)

print(
    f"Overall MAE         : {overall_mae:.3f} sec"
)

print(
    f"Missing share       : {missing_share*100:.3f}%"
)

print(
    "Missing branch      :",
    ACTIVE_MISSING_NAME,
)

print(
    "Main LGB iterations :",
    LGB_BEST_ITERS,
)

print(
    "Overall metric mode :",
    overall_metric_mode,
)

REFERENCE_V9_KNOWN = 227.04874547025202
REFERENCE_V9_OVERALL = 243.21867059251122

print(
    "Known gain vs V9:",
    f"{REFERENCE_V9_KNOWN - known_rmse:+.3f} sec"
)

print(
    "Overall gain vs V9:",
    f"{REFERENCE_V9_OVERALL - overall_rmse:+.3f} sec"
)

FROZEN_LGB_ITERS = int(
    LGB_BEST_ITERS
)

del lgb_model

gc.collect()

In [ ]:
#@title 12. V17 October diagnostics

print(
    "October validation rows:",
    f"{len(y_valid_all):,}"
)

print(
    "Known-anchor rows:",
    f"{valid_known_mask_full.sum():,}"
)

print(
    "Missing-anchor rows:",
    f"{valid_missing_mask_full.sum():,}"
)

print()
print(
    f"Known-anchor RMSE: {known_rmse:.3f} sec"
)

print(
    f"Known-anchor MAE : {known_mae:.3f} sec"
)

print(
    f"Active missing benchmark RMSE: {ACTIVE_MISSING_RMSE:.3f} sec"
)

print(
    f"Overall RMSE: {overall_rmse:.3f} sec"
)

# Exact full-row residual distribution only when row-level missing predictions
# are available. Otherwise the overall metric above is analytically combined.
if (
    "pred_valid_all" in globals()
    and len(
        pred_valid_all
    )
    == len(
        y_valid_all
    )
):
    error_all = (
        y_valid_all.astype(
            np.float64
        )
        - pred_valid_all
    )

    print()
    print(
        "95th percentile absolute error:",
        np.quantile(
            np.abs(
                error_all
            ),
            0.95,
        )
    )

    print(
        "99th percentile absolute error:",
        np.quantile(
            np.abs(
                error_all
            ),
            0.99,
        )
    )

else:
    print()
    print(
        "Row-level full validation vector is unavailable because the frozen "
        "V9 missing benchmark has metrics but no current row-level predictions."
    )

In [ ]:
#@title 12b. V17.2 FINAL ONE-RUN diagnostics

print(
    "Single LightGBM residual RMSE:",
    f"{lgb_rmse:.3f} sec"
)

print(
    "Single LightGBM iterations:",
    LGB_BEST_ITERS
)

print()
print(
    "Top V17.2 gain features:"
)

display(
    LGB_GAIN_IMPORTANCE
    .head(
        60
    )
    .rename(
        "gain"
    )
    .rename_axis(
        "feature"
    )
    .reset_index()
)

# Anchor source quality
if "nm_anchor_source" in X_valid.columns:
    source_values = (
        X_valid["nm_anchor_source"]
        .astype("string")
        .fillna("__MISSING__")
        .astype(str)
        .to_numpy()
    )

    rows = []

    for source_name in pd.unique(
        source_values
    ):
        mask = (
            source_values
            == source_name
        )

        if not mask.any():
            continue

        rows.append({
            "anchor_source": source_name,
            "rows": int(
                mask.sum()
            ),
            "anchor_only_rmse": float(
                np.sqrt(
                    np.mean(
                        np.square(
                            y_valid_residual[
                                mask
                            ].astype(
                                np.float64
                            )
                        )
                    )
                )
            ),
            "corrected_rmse": float(
                mean_squared_error(
                    y_valid_residual[
                        mask
                    ],
                    pred_known_residual[
                        mask
                    ],
                )
                ** 0.5
            ),
        })

    anchor_source_diagnostics = (
        pd.DataFrame(
            rows
        )
        .sort_values(
            "corrected_rmse"
        )
    )

    print()
    print(
        "NM anchor-source diagnostics:"
    )

    display(
        anchor_source_diagnostics
    )

print()
print(
    "Active missing branch:",
    ACTIVE_MISSING_NAME,
)

print(
    "Active missing RMSE:",
    ACTIVE_MISSING_RMSE,
)

if "raw_peer_scores" in globals():
    display(
        raw_peer_scores
    )

if "peer_blend_scores" in globals():
    display(
        peer_blend_scores.head(
            15
        )
    )

In [ ]:
#@title 13. V17.2 FINAL ONE-RUN frozen settings

print(
    "Main LightGBM max iterations:",
    LGB_MAX_ITERS
)

print(
    "Main LightGBM selected iterations:",
    FROZEN_LGB_ITERS
)

print(
    "Missing LightGBM selected iterations:",
    PEER_FIXED_ITERS
)

print(
    "Best missing blend config:",
    BEST_PEER_CONFIG
)

print(
    "Pseudo-missing rows:",
    PSEUDO_MISSING_ROWS
)

## Full-2025 production fit + ranking inference

When `RUN_FINAL=True`, the next cell rebuilds the feature pipeline on all 2025
training data, fits the frozen V17 models, predicts every ranking departure,
and creates `rank_pred`.

This replaces the stale legacy December/single-model path.

In [ ]:
#@title 14. V17.2 FINAL ONE-RUN full-2025 production fit + ranking prediction — guarded

if not RUN_FINAL:
    print(
        "Production fit LOCKED. Set RUN_FINAL=True only after accepting "
        "the V17 development result."
    )

else:
    print()
    print(
        "=" * 64
    )

    print(
        "V17.2 FINAL ONE-RUN FULL-2025 PRODUCTION PIPELINE"
    )

    print(
        "=" * 64
    )

    report_ram(
        "production start"
    )

    # ========================================================
    # 1. Reload complete raw data so production is independent
    #    of objects freed during development.
    # ========================================================
    prod_parts = []

    for i, fname in enumerate(
        TRAIN_FILES,
        start=1,
    ):
        part = pd.read_parquet(
            RAW_DIR
            / fname
        )

        prod_parts.append(
            part
        )

        print(
            f"production load {i:02d}/12: {fname} "
            f"({len(part):,} rows)"
        )

    prod_train_all = pd.concat(
        prod_parts,
        ignore_index=True,
    )

    del prod_parts
    gc.collect()

    prod_rank_all = pd.read_parquet(
        RAW_DIR
        / RANKING_FILE
    )

    # ========================================================
    # 2. Batch traffic + dynamic queue features.
    # ========================================================
    print()
    print(
        "Production traffic context..."
    )

    prod_train_all = build_traffic_context(
        prod_train_all
    )

    prod_rank_all = build_traffic_context(
        prod_rank_all
    )

    prod_train_dep = (
        prod_train_all.loc[
            prod_train_all[
                "PHASE_mvt"
            ].eq(
                "DEP"
            )
            & prod_train_all[
                TARGET
            ].notna()
        ]
        .copy()
    )

    prod_rank_dep = (
        prod_rank_all.loc[
            prod_rank_all[
                "PHASE_mvt"
            ].eq(
                "DEP"
            )
        ]
        .copy()
    )

    if len(
        prod_rank_dep
    ) != len(
        template
    ):
        raise ValueError(
            "Production ranking departure count does not match template."
        )

    print(
        "Production train departures:",
        f"{len(prod_train_dep):,}"
    )

    print(
        "Production ranking departures:",
        f"{len(prod_rank_dep):,}"
    )

    print()
    print(
        "Production dynamic queue features..."
    )

    prod_train_dep = build_dynamic_features_for_departures(
        prod_train_all,
        prod_train_dep,
    )

    prod_rank_dep = build_dynamic_features_for_departures(
        prod_rank_all,
        prod_rank_dep,
    )

    del (
        prod_train_all,
        prod_rank_all,
    )

    gc.collect()

    report_ram(
        "after production dynamic features"
    )

    # ========================================================
    # 3. Full-year historical prior + static graph state.
    # ========================================================
    PROD_GLOBAL_MEDIAN = float(
        prod_train_dep[
            TARGET
        ].median()
    )

    PROD_PRIOR_STATE = fit_target_priors(
        prod_train_dep
    )

    PROD_MISSING_STATE = fit_missing_fallback_state(
        prod_train_dep
    )

    prod_train_dep = apply_target_priors(
        prod_train_dep,
        PROD_PRIOR_STATE,
    )

    prod_rank_dep = apply_target_priors(
        prod_rank_dep,
        PROD_PRIOR_STATE,
    )

    PROD_GRAPH_STATE = build_graph_embedding_state(
        prod_train_dep,
        dim=GRAPH_EMBED_DIM,
        min_edge_count=GRAPH_MIN_EDGE_COUNT,
        random_state=SEED + 1717,
    )

    prod_train_dep = apply_graph_embeddings(
        prod_train_dep,
        PROD_GRAPH_STATE,
    )

    prod_rank_dep = apply_graph_embeddings(
        prod_rank_dep,
        PROD_GRAPH_STATE,
    )

    prod_train_u = add_unified_baseline(
        prod_train_dep,
        PROD_GLOBAL_MEDIAN,
    )

    prod_rank_u = add_unified_baseline(
        prod_rank_dep,
        PROD_GLOBAL_MEDIAN,
    )

    prod_train_known_mask = (
        prod_train_u[
            "baseline_is_nm"
        ]
        .eq(
            1
        )
        .to_numpy()
    )

    prod_rank_known_mask = (
        prod_rank_u[
            "baseline_is_nm"
        ]
        .eq(
            1
        )
        .to_numpy()
    )

    prod_rank_missing_mask = (
        ~prod_rank_known_mask
    )

    print(
        "Production ranking NM availability:",
        f"{prod_rank_known_mask.mean()*100:.3f}%"
    )

    # Full ordered ranking IDs preserved before any in-place selector.
    prod_rank_ids = (
        prod_rank_dep[
            "MVT_ID_mvt"
        ]
        .to_numpy(
            copy=True
        )
    )

    # ========================================================
    # 4. Production missing-anchor model.
    # ========================================================
    prod_missing_pred = None

    if prod_rank_missing_mask.any():
        print()
        print(
            "=== PRODUCTION AUGMENTED MISSING-ANCHOR MODEL ==="
        )

        (
            prod_missing_train_df,
            prod_peer_sample_weight,
            prod_n_real_missing,
            prod_n_synth_missing,
        ) = prepare_augmented_missing_training(
            prod_train_u,
            pseudo_rows=PSEUDO_MISSING_ROWS,
            random_state=SEED + 1720,
            audit=False,
        )

        prod_missing_rank_df = (
            attach_peer_features_for_targets(
                prod_rank_u,
                prod_rank_missing_mask,
                audit=True,
            )
        )

        add_missing_prior_features_inplace(
            prod_missing_train_df,
            prod_missing_rank_df,
        )

        prod_pseudo_train = (
            pd.to_numeric(
                prod_missing_train_df[
                    "peer_pseudo_anchor_sec"
                ],
                errors="coerce",
            )
            .fillna(
                PROD_GLOBAL_MEDIAN
            )
            .to_numpy(
                dtype=np.float32,
                copy=True,
            )
        )

        prod_pseudo_rank = (
            pd.to_numeric(
                prod_missing_rank_df[
                    "peer_pseudo_anchor_sec"
                ],
                errors="coerce",
            )
            .fillna(
                PROD_GLOBAL_MEDIAN
            )
            .to_numpy(
                dtype=np.float64,
                copy=True,
            )
        )

        prod_y_missing = (
            prod_missing_train_df[
                TARGET
            ]
            .astype(
                "float32"
            )
            .to_numpy(
                copy=True
            )
        )

        prod_y_missing_corr = (
            prod_y_missing
            - prod_pseudo_train
        ).astype(
            np.float32
        )

        X_prod_missing, PROD_PEER_FEATURES, PROD_PEER_CATS = (
            make_original_method_b_plus_graph_inplace(
                prod_missing_train_df
            )
        )

        PROD_PEER_FEATURES = list(
            X_prod_missing.columns
        )

        prod_peer_drop = [
            c
            for c in PROD_PEER_FEATURES
            if (
                c in PEER_DROP_EXACT
                or (
                    c.startswith(
                        PEER_DROP_PREFIXES
                    )
                    and not c.startswith(
                        "mp_"
                    )
                )
            )
        ]

        X_prod_missing.drop(
            columns=prod_peer_drop,
            inplace=True,
            errors="ignore",
        )

        PROD_PEER_FEATURES = list(
            X_prod_missing.columns
        )

        PROD_PEER_CATS = [
            c
            for c in PROD_PEER_CATS
            if c in PROD_PEER_FEATURES
        ]

        if (
            "peer_selected_level"
            in PROD_PEER_FEATURES
            and "peer_selected_level"
            not in PROD_PEER_CATS
        ):
            PROD_PEER_CATS.append(
                "peer_selected_level"
            )

        X_rank_missing, _, _ = (
            make_original_method_b_plus_graph_inplace(
                prod_missing_rank_df
            )
        )

        X_rank_missing.drop(
            columns=[
                c
                for c in X_rank_missing.columns
                if c not in PROD_PEER_FEATURES
            ],
            inplace=True,
            errors="ignore",
        )

        for c in PROD_PEER_FEATURES:
            if c not in X_rank_missing.columns:
                X_rank_missing[
                    c
                ] = np.nan

        X_rank_missing = X_rank_missing.reindex(
            columns=PROD_PEER_FEATURES
        )

        for c in PROD_PEER_CATS:
            tr = (
                X_prod_missing[
                    c
                ]
                .astype(
                    "string"
                )
                .fillna(
                    "__MISSING__"
                )
            )

            categories = pd.Index(
                pd.unique(
                    tr
                )
            )

            X_prod_missing[
                c
            ] = pd.Categorical(
                tr,
                categories=categories,
            )

            ra = (
                X_rank_missing[
                    c
                ]
                .astype(
                    "string"
                )
                .fillna(
                    "__MISSING__"
                )
            )

            X_rank_missing[
                c
            ] = pd.Categorical(
                ra,
                categories=categories,
            )

        prod_peer_model = LGBMRegressor(
            objective="regression",
            metric="rmse",
            n_estimators=max(
                200,
                int(
                    PEER_FIXED_ITERS
                ),
            ),
            learning_rate=0.020,
            num_leaves=48,
            max_depth=-1,
            min_child_samples=120,
            subsample=0.90,
            subsample_freq=1,
            colsample_bytree=0.82,
            reg_alpha=0.10,
            reg_lambda=24.0,
            max_bin=255,
            device_type="cpu",
            random_state=SEED + 1721,
            n_jobs=-1,
            verbosity=-1,
        )

        prod_peer_model.fit(
            X_prod_missing,
            prod_y_missing_corr,
            sample_weight=prod_peer_sample_weight,
            categorical_feature=PROD_PEER_CATS,
            callbacks=[
                lgb_log_evaluation(
                    250
                )
            ],
        )

        prod_peer_corr = (
            prod_peer_model.predict(
                X_rank_missing
            )
            .astype(
                np.float64
            )
        )

        prod_peer_corrected = (
            prod_pseudo_rank
            + prod_peer_corr
        )

        prod_general_prior = (
            pd.to_numeric(
                prod_rank_u.loc[
                    prod_rank_missing_mask,
                    "baseline_sec",
                ],
                errors="coerce",
            )
            .fillna(
                PROD_GLOBAL_MEDIAN
            )
            .to_numpy(
                dtype=np.float64,
            )
        )

        prod_hier_mean = predict_missing_hierarchy(
            prod_rank_u.loc[
                prod_rank_missing_mask
            ],
            PROD_MISSING_STATE,
            statistic="mean",
        ).astype(
            np.float64
        )

        prod_hist = (
            0.50
            * prod_general_prior
            + 0.50
            * prod_hier_mean
        )

        cfg = dict(
            BEST_PEER_CONFIG
        )

        prod_missing_pred = (
            float(
                cfg.get(
                    "w_corrected",
                    0.90,
                )
            )
            * prod_peer_corrected
            + float(
                cfg.get(
                    "w_peer_raw",
                    0.00,
                )
            )
            * prod_pseudo_rank
            + float(
                cfg.get(
                    "w_hist",
                    0.10,
                )
            )
            * prod_hist
        )

        clip_max = cfg.get(
            "clip_max",
            None,
        )

        if clip_max is not None:
            prod_missing_pred = np.clip(
                prod_missing_pred,
                0.0,
                float(
                    clip_max
                ),
            )

        prod_missing_pred = np.clip(
            prod_missing_pred,
            0.0,
            None,
        )

        print(
            "Production missing predictions:",
            f"{len(prod_missing_pred):,}"
        )

        del (
            X_prod_missing,
            X_rank_missing,
            prod_peer_model,
            prod_missing_train_df,
            prod_missing_rank_df,
            prod_peer_sample_weight,
            prod_y_missing,
            prod_y_missing_corr,
            prod_pseudo_train,
            prod_pseudo_rank,
            prod_peer_corr,
            prod_peer_corrected,
            prod_general_prior,
            prod_hier_mean,
            prod_hist,
        )

        gc.collect()

        report_ram(
            "after production missing model"
        )

    # ========================================================
    # 5. Production known-anchor residual ensemble.
    # ========================================================
    print()
    print(
        "=== PRODUCTION KNOWN-ANCHOR SINGLE LIGHTGBM ==="
    )

    prod_train_known_df = (
        prod_train_u.loc[
            prod_train_known_mask
        ]
        .copy()
    )

    prod_rank_known_df = (
        prod_rank_u.loc[
            prod_rank_known_mask
        ]
        .copy()
    )

    prod_y_residual = (
        (
            prod_train_known_df[
                TARGET
            ]
            .astype(
                "float32"
            )
            - prod_train_known_df[
                "nm_anchor_sec"
            ]
            .astype(
                "float32"
            )
        )
        .to_numpy(
            dtype=np.float32,
            copy=True,
        )
    )

    prod_rank_anchor = (
        prod_rank_known_df[
            "nm_anchor_sec"
        ]
        .astype(
            "float64"
        )
        .to_numpy(
            copy=True
        )
    )

    X_prod_known, PROD_FEATURES, PROD_CATS = (
        make_original_method_b_plus_graph_inplace(
            prod_train_known_df
        )
    )

    PROD_FEATURES = list(
        X_prod_known.columns
    )

    PROD_CATS = [
        c
        for c in PROD_CATS
        if c in PROD_FEATURES
    ]

    X_rank_known, _, _ = (
        make_original_method_b_plus_graph_inplace(
            prod_rank_known_df
        )
    )

    X_rank_known.drop(
        columns=[
            c
            for c in X_rank_known.columns
            if c not in PROD_FEATURES
        ],
        inplace=True,
        errors="ignore",
    )

    for c in PROD_FEATURES:
        if c not in X_rank_known.columns:
            X_rank_known[
                c
            ] = np.nan

    X_rank_known = X_rank_known.reindex(
        columns=PROD_FEATURES
    )

    for c in PROD_CATS:
        tr = (
            X_prod_known[
                c
            ]
            .astype(
                "string"
            )
            .fillna(
                "__MISSING__"
            )
        )

        categories = pd.Index(
            pd.unique(
                tr
            )
        )

        X_prod_known[
            c
        ] = pd.Categorical(
            tr,
            categories=categories,
        )

        ra = (
            X_rank_known[
                c
            ]
            .astype(
                "string"
            )
            .fillna(
                "__MISSING__"
            )
        )

        X_rank_known[
            c
        ] = pd.Categorical(
            ra,
            categories=categories,
        )

        del (
            tr,
            ra,
            categories,
        )

    for c in PROD_FEATURES:
        if c in PROD_CATS:
            continue

        X_prod_known[
            c
        ] = pd.to_numeric(
            X_prod_known[
                c
            ],
            errors="coerce",
            downcast="float",
        )

        X_rank_known[
            c
        ] = pd.to_numeric(
            X_rank_known[
                c
            ],
            errors="coerce",
            downcast="float",
        )

    report_ram(
        "production known matrix ready"
    )


    print()
    print(
        "Training ONE production LightGBM..."
    )

    production_model = (
        make_v17_single_lgb_model(
            n_estimators=min(
                1200,
                int(
                    FROZEN_LGB_ITERS
                ),
            ),
            seed=SEED + 4100,
        )
    )

    production_model.fit(
        X_prod_known,
        prod_y_residual,
        categorical_feature=PROD_CATS,
        callbacks=[
            lgb_log_evaluation(
                200
            )
        ],
    )

    production_known_residual = (
        production_model.predict(
            X_rank_known
        )
        .astype(
            np.float64
        )
    )

    production_known_pred = np.clip(
        prod_rank_anchor
        + production_known_residual,
        0.0,
        None,
    )

    print(
        "Production main LGB iterations:",
        min(
            1200,
            int(
                FROZEN_LGB_ITERS
            ),
        )
    )

    del production_model
    gc.collect()

    # ========================================================
    # 6. Reassemble exact ranking order.
    # ========================================================
    rank_pred = np.full(
        len(
            prod_rank_dep
        ),
        np.nan,
        dtype=np.float64,
    )

    rank_pred[
        prod_rank_known_mask
    ] = production_known_pred

    if prod_rank_missing_mask.any():
        if prod_missing_pred is None:
            raise RuntimeError(
                "Missing ranking rows exist but production missing predictions "
                "were not generated."
            )

        rank_pred[
            prod_rank_missing_mask
        ] = prod_missing_pred

    if not np.isfinite(
        rank_pred
    ).all():
        bad = int(
            (
                ~np.isfinite(
                    rank_pred
                )
            ).sum()
        )

        raise RuntimeError(
            f"Production rank_pred contains {bad:,} invalid values."
        )

    # Provide the exact object expected by submission packaging.
    ranking_dep = prod_rank_dep

    if not np.array_equal(
        ranking_dep[
            "MVT_ID_mvt"
        ].to_numpy(),
        prod_rank_ids,
    ):
        raise RuntimeError(
            "Production ranking order changed unexpectedly."
        )

    print()
    print(
        "=" * 64
    )

    print(
        "V17 PRODUCTION PREDICTIONS READY"
    )

    print(
        "=" * 64
    )

    print(
        "Ranking rows       :",
        f"{len(rank_pred):,}"
    )

    print(
        "Known-anchor rows  :",
        f"{prod_rank_known_mask.sum():,}"
    )

    print(
        "Missing-anchor rows:",
        f"{prod_rank_missing_mask.sum():,}"
    )

    print(
        "Prediction min/max :",
        f"{rank_pred.min():.1f}",
        "/",
        f"{rank_pred.max():.1f}",
    )

    display(
        pd.Series(
            rank_pred,
            name="prediction",
        ).describe(
            percentiles=[
                .001,
                .01,
                .05,
                .50,
                .95,
                .99,
                .999,
            ]
        )
    )

    del (
        X_prod_known,
        X_rank_known,
        prod_train_known_df,
        prod_rank_known_df,
        production_known_residual,
        production_known_pred,
        prod_y_residual,
        prod_rank_anchor,
        prod_train_u,
        prod_rank_u,
        prod_train_dep,
    )

    gc.collect()

    report_ram(
        "production complete"
    )

## Official submission packaging

When `RUN_FINAL=True`, the production cell above creates `rank_pred` and
`ranking_dep`. The remaining cells validate the exact schema/order, save the
official parquet, reload it, and print the generated filename/path/bucket.

In [ ]:
#@title 15. Production prediction handoff — V17.2 FINAL ONE-RUN

if not RUN_FINAL:
    print("Submission export LOCKED. Set RUN_FINAL=True when ready.")
elif "rank_pred" not in globals():
    print(
        "rank_pred is not available yet. Run the V17.2 FINAL ONE-RUN 5-seed "
        "LightGBM production inference for ranking_dep first."
    )
else:
    _rank_pred = np.asarray(rank_pred, dtype=np.float64).reshape(-1)

    if len(_rank_pred) != len(ranking_dep):
        raise ValueError(
            f"rank_pred length {len(_rank_pred):,} != "
            f"ranking departures {len(ranking_dep):,}"
        )

    if not np.isfinite(_rank_pred).all():
        raise ValueError("rank_pred contains NaN/inf values.")

    print("✓ rank_pred valid:", f"{len(_rank_pred):,} rows")

In [ ]:
#@title 16. Build official submission — guarded

if not RUN_FINAL:
    print("Submission build skipped because RUN_FINAL=False.")
elif "rank_pred" not in globals():
    print("Submission build skipped: rank_pred is missing.")
else:
    _rank_pred = np.asarray(rank_pred, dtype=np.float64).reshape(-1)

    prediction_frame = pd.DataFrame({
        "MVT_ID_mvt": ranking_dep["MVT_ID_mvt"].to_numpy(copy=True),
        "TAXITIME_SEC_mvt": _rank_pred,
    })

    if prediction_frame["MVT_ID_mvt"].duplicated().any():
        raise ValueError("Duplicate MVT_ID_mvt in ranking departures.")

    submission = template[["MVT_ID_mvt"]].copy()

    if submission["MVT_ID_mvt"].duplicated().any():
        raise ValueError("Duplicate MVT_ID_mvt in official template.")

    submission = submission.merge(
        prediction_frame,
        on="MVT_ID_mvt",
        how="left",
        sort=False,
        validate="one_to_one",
    )

    required_cols = ["MVT_ID_mvt", "TAXITIME_SEC_mvt"]

    if list(submission.columns) != required_cols:
        raise ValueError(f"Wrong submission columns: {list(submission.columns)}")

    if len(submission) != len(template) or len(submission) != len(ranking_dep):
        raise ValueError("Submission row count mismatch.")

    if not np.array_equal(
        submission["MVT_ID_mvt"].to_numpy(),
        template["MVT_ID_mvt"].to_numpy(),
    ):
        raise ValueError("Submission ID order does not match template.")

    if submission["TAXITIME_SEC_mvt"].isna().any():
        raise ValueError("Submission contains missing predictions.")

    _p = submission["TAXITIME_SEC_mvt"].to_numpy(dtype=np.float64)

    if not np.isfinite(_p).all():
        raise ValueError("Submission contains NaN/inf predictions.")

    submission["TAXITIME_SEC_mvt"] = np.clip(
        _p, 0.0, None
    ).astype("float32")

    print("✓ OFFICIAL SUBMISSION VALIDATION PASSED")
    print("Rows:", f"{len(submission):,}")
    print("Columns:", list(submission.columns))
    display(submission.head())

In [ ]:
#@title 17. Final pre-save checks — guarded

if not RUN_FINAL:
    print("Pre-save checks skipped because RUN_FINAL=False.")
elif "submission" not in globals():
    print("Pre-save checks skipped: submission is not built.")
else:
    checks = {
        "exact_columns": list(submission.columns)
            == ["MVT_ID_mvt", "TAXITIME_SEC_mvt"],
        "exact_row_count": len(submission) == len(template),
        "exact_id_order": np.array_equal(
            submission["MVT_ID_mvt"].to_numpy(),
            template["MVT_ID_mvt"].to_numpy(),
        ),
        "unique_ids": not submission["MVT_ID_mvt"].duplicated().any(),
        "all_predictions_present": submission["TAXITIME_SEC_mvt"].notna().all(),
        "all_predictions_finite": np.isfinite(
            submission["TAXITIME_SEC_mvt"].to_numpy(dtype=np.float64)
        ).all(),
    }

    for key, ok in checks.items():
        print(f"{'✓' if ok else '✗'} {key}")

    failed = [k for k, ok in checks.items() if not ok]
    if failed:
        raise RuntimeError("Submission validation failed: " + ", ".join(failed))

    print("✓ Structurally ready for challenge submission.")

In [ ]:
#@title 18. Save official challenge file — guarded

if not RUN_FINAL:
    print("Submission save skipped because RUN_FINAL=False.")
elif "submission" not in globals():
    print("Submission save skipped: submission is not built.")
else:
    if SUBMISSION_SUFFIX == "final":
        OFFICIAL_SUBMISSION_NAME = f"{TEAM_NAME}_final.parquet"
    else:
        if int(SUBMISSION_VERSION) < 1:
            raise ValueError("SUBMISSION_VERSION must be >= 1.")
        OFFICIAL_SUBMISSION_NAME = (
            f"{TEAM_NAME}_v{int(SUBMISSION_VERSION)}.parquet"
        )

    OUTPUT = OUTPUT_DIR / OFFICIAL_SUBMISSION_NAME
    submission.to_parquet(OUTPUT, index=False)

    # Reload the actual saved file and verify it again.
    _saved = pd.read_parquet(OUTPUT)

    if list(_saved.columns) != ["MVT_ID_mvt", "TAXITIME_SEC_mvt"]:
        raise RuntimeError("Saved parquet schema is incorrect.")

    if len(_saved) != len(template):
        raise RuntimeError("Saved parquet row count is incorrect.")

    if not np.array_equal(
        _saved["MVT_ID_mvt"].to_numpy(),
        template["MVT_ID_mvt"].to_numpy(),
    ):
        raise RuntimeError("Saved parquet ID order differs from template.")

    if not np.isfinite(
        _saved["TAXITIME_SEC_mvt"].to_numpy(dtype=np.float64)
    ).all():
        raise RuntimeError("Saved parquet has invalid predictions.")

    print("=" * 48)
    print("OFFICIAL SUBMISSION FILE READY")
    print("=" * 48)
    print("Stage:", "FINAL" if SUBMISSION_SUFFIX == "final" else "RANKING")
    print("Filename:", OUTPUT.name)
    print("Path:", OUTPUT)
    print("Rows:", f"{len(_saved):,}")
    print("Size:", f"{OUTPUT.stat().st_size/1024**2:.2f} MiB")
    print("✓ Reload validation passed.")

    del _saved

In [ ]:
#@title 19. Final submission summary + download

if RUN_FINAL and "OUTPUT" in globals() and OUTPUT.exists():
    stage_name = (
        "FINAL"
        if SUBMISSION_SUFFIX == "final"
        else "RANKING"
    )

    print()
    print("=" * 60)
    print("SUBMISSION ARTIFACT GENERATED")
    print("=" * 60)
    print("Team name        :", TEAM_NAME)
    print("Stage            :", stage_name)
    print("Filename         :", OUTPUT.name)
    print("Local file path  :", str(OUTPUT))
    print("Submission bucket:", SUBMISSION_BUCKET)
    print("Rows             :", f"{len(submission):,}")
    print("Columns          :", list(submission.columns))
    print("File size        :", f"{OUTPUT.stat().st_size/1024**2:.2f} MiB")
    print()
    print("Upload this exact file to your OpenSky bucket:")
    print(f"  bucket : {SUBMISSION_BUCKET}")
    print(f"  file   : {OUTPUT.name}")
    print()
    print("✓ FINAL RUN COMPLETE — submission file is ready.")

    try:
        from google.colab import files
        files.download(str(OUTPUT))
    except ImportError:
        print()
        print("Colab download helper unavailable.")
        print("Use the local file path shown above.")

else:
    print()
    print("=" * 60)
    print("NO SUBMISSION FILE GENERATED YET")
    print("=" * 60)
    print("Expected team name :", TEAM_NAME)
    print("Submission bucket  :", SUBMISSION_BUCKET)

    if SUBMISSION_SUFFIX == "final":
        expected_name = f"{TEAM_NAME}_final.parquet"
    else:
        expected_name = (
            f"{TEAM_NAME}_v{int(SUBMISSION_VERSION)}.parquet"
        )

    print("Expected filename  :", expected_name)
    print()
    print(
        "Set RUN_FINAL=True, create rank_pred with the V15 production "
        "inference, then run the submission cells again."
    )

## V17.2 FINAL ONE-RUN architecture

```text
KNOWN NM ANCHOR
        |
V17 operational / queue / recurrence features
        |
ONE LightGBM
≤ 1200 trees
        |
NM anchor + residual


MISSING NM ANCHOR
        |
real + synthetic missing training rows
        |
peer-imputed pseudo anchor
        |
ONE LightGBM correction
≤ 1200 trees
```

The full-2025 production path also uses only one main LightGBM.